# Minimum SAE Circuit Discovery v011: Invariant Feature Circuit Benchmark

V10 treated the failure as a ranking/calibration problem. V11 treats it as a scientific object: a sparse SAE circuit is not convincing unless it is stable across nuisance environments before the final held-out splits are touched.

The experiment builds separate feature rankings on each optimization environment, aggregates only features that repeatedly rank well, and evaluates whether these invariant feature banks generalize better than pooled-train and single-environment masks.

The key tests are:

- do environment-stable rankings beat pooled global calibrated rankings at the same K;
- do single-environment rankings overfit their source split and fail held-out names/templates;
- does counterfactual name-family variance predict final held-out failure;
- is the minimum faithful circuit larger once invariance, not just average train faithfulness, is required.

Final held-out templates/names remain untouched for selection.

## 1. Colab Setup

Run this notebook in a GPU runtime. The smoke cells are small; the full baseline comparison runs several top-K sweeps and is intended for Colab Pro+.


In [ ]:
%pip install -q "sae-lens>=6,<7" pandas matplotlib tqdm


In [ ]:
import json
import os
import random
import shutil
from collections import defaultdict
from contextlib import nullcontext
from datetime import datetime
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from sae_lens import SAE, HookedSAETransformer
from tqdm.auto import tqdm

SEED = 12345
random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("This notebook expects a CUDA GPU. In Colab, enable Runtime -> Change runtime type -> GPU.")

device = "cuda"
torch.set_grad_enabled(False)

props = torch.cuda.get_device_properties(0)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")
print(f"PyTorch: {torch.__version__}")


## 2. Versioned Drive Output

All v11 artifacts go to a separate Google Drive version folder. The notebook keeps timestamped trials, saves environment-stability diagnostics, and mirrors completed artifacts to `latest/<RUN_VERSION>/`.

In [ ]:
MOUNT_DRIVE = True
PROJECT_DIR_NAME = "minimum_sae_circuit_discovery"
RUN_VERSION = "v011_invariant_feature_circuit_benchmark"
TRIAL_ID = os.environ.get("TRIAL_ID") or datetime.now().strftime("trial_%Y%m%d_%H%M%S")
RUN_STARTED_AT = datetime.now().astimezone().isoformat(timespec="seconds")

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and MOUNT_DRIVE:
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive") / PROJECT_DIR_NAME
else:
    PROJECT_ROOT = Path.cwd() / f"{PROJECT_DIR_NAME}_outputs"

CACHE_DIR = PROJECT_ROOT / "cache" / RUN_VERSION
OUTPUT_DIR = PROJECT_ROOT / "runs" / RUN_VERSION / TRIAL_ID
LATEST_DIR = PROJECT_ROOT / "latest" / RUN_VERSION
for directory in (CACHE_DIR, OUTPUT_DIR, LATEST_DIR):
    directory.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "gpt2-small"
SAE_RELEASE = "gpt2-small-res-jb"
TARGET_LAYER = 8
EXPECTED_D_SAE = 24_576

SMOKE_N_PROMPTS = 8
ENV_TRAIN_N_PROMPTS = 128
COUNTERFACTUAL_TRAIN_FAMILIES = 96
COUNTERFACTUAL_EVAL_FAMILIES = 48
COUNTERFACTUAL_FAMILY_SIZE = 4
COUNTERFACTUAL_TRAIN_N_PROMPTS = COUNTERFACTUAL_TRAIN_FAMILIES * COUNTERFACTUAL_FAMILY_SIZE
TRAIN_N_PROMPTS = ENV_TRAIN_N_PROMPTS * 4 + COUNTERFACTUAL_TRAIN_N_PROMPTS
OPTIMIZATION_N_PROMPTS = ENV_TRAIN_N_PROMPTS
EVAL_N_PROMPTS = 128
N_PROMPTS = TRAIN_N_PROMPTS
BATCH_SIZE = 16
TRAIN_BATCH_SIZE = 8
COUNTERFACTUAL_BATCH_FAMILIES = 4
ANSWER_ATTRIBUTION_BATCH_SIZE = 4
ANSWER_ATTRIBUTION_PRIOR_WEIGHT = 0.25
ANSWER_ATTRIBUTION_SCORE_EPS = 1e-8

GLOBAL_K_VALUES = [200, 500, 1_000, 2_000, 4_000, 8_000]
ROLE_K_VALUES = [100, 200, 500, 800, 1_000, 1_500, 2_000]
SMOKE_GLOBAL_K_VALUES = [20, 100]
SMOKE_ROLE_K_VALUES = [20, 100]

TOP_GLOBAL_FEATURE_POOL = 8_000
GLOBAL_SOFT_CANDIDATE_FEATURES = 8_000
RUN_GLOBAL_SOFT_MASK = False
GLOBAL_SOFT_LAMBDAS = [0.0003, 0.001, 0.003, 0.01]
GLOBAL_SOFT_LAMBDA = GLOBAL_SOFT_LAMBDAS[1]
GLOBAL_SOFT_STEPS = 220
GLOBAL_SOFT_LR = 0.08
GLOBAL_SOFT_MAX_GAIN = 1.50
GLOBAL_ATTRIBUTION_PRIOR_WEIGHT = 0.15
GLOBAL_GAIN_ANCHOR_WEIGHT = 0.05
GLOBAL_SOFT_TOP_K_VALUES = [500, 1_000, 1_500, 2_000, 3_000, 4_000]
GLOBAL_SOFT_THRESHOLD_VALUES = [0.35, 0.50, 0.70, 0.90]
ROBUST_WORST_ENV_WEIGHT = 1.5
ROBUST_VARIANCE_WEIGHT = 0.75
ROBUST_EXAMPLE_RATIO_WEIGHT = 0.50
COUNTERFACTUAL_RATIO_WEIGHT = 1.00
COUNTERFACTUAL_FAMILY_MEAN_WEIGHT = 1.00
COUNTERFACTUAL_NAME_VARIANCE_WEIGHT = 3.00
COUNTERFACTUAL_WORST_FAMILY_WEIGHT = 1.00
RUN_GROUP_DIAGNOSTICS = True
RUN_COUNTERFACTUAL_DIAGNOSTICS = True
GROUP_DIAGNOSTIC_TOP_N = 8
MIN_GROUP_PROMPTS = 10

FAITHFULNESS_BAND_LOW = 0.95
FAITHFULNESS_BAND_HIGH = 1.05
RANDOM_CONTROL_REPEATS = 3
RANDOM_CONTROL_K_VALUES = [500, 2_000]

STABILITY_SOURCE_RANKINGS = [
    "global_activation_mean_abs",
    "global_answer_gradient_abs",
    "global_answer_gradient_support",
    "global_calibrated_rescue_union",
    "global_calibrated_borda_activation_answer",
]
STABILITY_POOL_K = 8_000
STABILITY_VOTE_TOP_K = 1_000
STABILITY_TOP_K_VALUES = [200, 500, 1_000, 2_000, 4_000]
STABILITY_SUMMARY_TOP_N = 75
INCLUDE_SINGLE_ENVIRONMENT_RANKINGS = True
VALIDATION_GLOBAL_BASELINES = [
    "global_activation_mean_abs",
    "global_answer_gradient_abs",
    "global_answer_gradient_support",
    "global_calibrated_rescue_union",
    "global_calibrated_borda_activation_answer",
    "global_stability_vote_calibrated_rescue_union",
    "global_stability_worst_calibrated_rescue_union",
    "global_stability_lowvar_calibrated_rescue_union",
]


VALIDATION_SPLIT_SEEDS = {
    "train_core": SEED,
    "train_template_aug": SEED + 11,
    "train_name_aug": SEED + 22,
    "train_crossed_aug": SEED + 33,
    "name_cf_train": SEED + 44,
    "heldout_name_families": SEED + 505,
    "heldout_template_name_families": SEED + 606,
    "id_test": SEED + 101,
    "heldout_templates": SEED + 202,
    "heldout_names": SEED + 303,
    "heldout_templates_names": SEED + 404,
}
OPTIMIZATION_ENV_NAMES = ["train_core", "train_template_aug", "train_name_aug", "train_crossed_aug", "name_cf_train"]
FINAL_VALIDATION_SPLITS = ["id_test", "heldout_templates", "heldout_names", "heldout_templates_names"]

TRAIN_STATS_CACHE_PATH = CACHE_DIR / f"pooled_train_role_feature_stats_layer{TARGET_LAYER}_ioi.pt"
ANSWER_DIRECTION_STATS_CACHE_PATH = CACHE_DIR / f"pooled_train_answer_direction_role_feature_stats_layer{TARGET_LAYER}_ioi.pt"
SMOKE_STATS_CACHE_PATH = CACHE_DIR / f"smoke_role_feature_stats_layer{TARGET_LAYER}_ioi.pt"
SMOKE_ANSWER_DIRECTION_STATS_CACHE_PATH = CACHE_DIR / f"smoke_answer_direction_role_feature_stats_layer{TARGET_LAYER}_ioi.pt"
SMOKE_RESULTS_CSV_PATH = OUTPUT_DIR / "smoke_invariant_feature_results.csv"
RESULTS_CSV_PATH = OUTPUT_DIR / "invariant_feature_validation_results.csv"
SUMMARY_CSV_PATH = OUTPUT_DIR / "invariant_feature_validation_summary.csv"
INVARIANT_TRACE_CSV_PATH = OUTPUT_DIR / "invariant_feature_stability_trace.csv"
MASKS_PATH = OUTPUT_DIR / "selected_masks.pt"
SMOKE_PLOT_PATH = OUTPUT_DIR / "smoke_invariant_feature_pareto.png"
PARETO_PLOT_PATH = OUTPUT_DIR / "invariant_feature_validation_pareto.png"
HEATMAP_PLOT_PATH = OUTPUT_DIR / "invariant_feature_generalization_heatmap.png"
GROUP_DIAGNOSTIC_CSV_PATH = OUTPUT_DIR / "invariant_feature_template_group_diagnostics.csv"
GROUP_DIAGNOSTIC_HEATMAP_PATH = OUTPUT_DIR / "invariant_feature_template_group_heatmap.png"
COUNTERFACTUAL_DIAGNOSTIC_CSV_PATH = OUTPUT_DIR / "invariant_feature_family_diagnostics.csv"
COUNTERFACTUAL_DIAGNOSTIC_HEATMAP_PATH = OUTPUT_DIR / "invariant_feature_family_heatmap.png"
MANIFEST_PATH = OUTPUT_DIR / "run_manifest.json"

ROLE_NAMES = ["bos", "subject_first", "indirect_object", "subject_repeat", "place", "object", "final_prediction", "other"]
ROLE_TO_ID = {name: idx for idx, name in enumerate(ROLE_NAMES)}
CORE_ROLE_NAMES = ["bos", "subject_first", "indirect_object", "subject_repeat", "final_prediction"]
CORE_ROLE_IDS = torch.tensor([ROLE_TO_ID[name] for name in CORE_ROLE_NAMES], dtype=torch.long)

def hook_name_for_layer(layer):
    return f"blocks.{layer}.hook_resid_pre"

def sae_id_for_layer(layer):
    return hook_name_for_layer(layer)

def lambda_tag(lambda_size):
    return str(lambda_size).replace("-", "m").replace(".", "_")

def write_run_manifest(status, extra=None):
    manifest = {
        "status": status,
        "run_started_at": RUN_STARTED_AT,
        "run_updated_at": datetime.now().astimezone().isoformat(timespec="seconds"),
        "run_version": RUN_VERSION,
        "trial_id": TRIAL_ID,
        "seed": SEED,
        "model_name": MODEL_NAME,
        "sae_release": SAE_RELEASE,
        "target_layer": TARGET_LAYER,
        "env_train_n_prompts": ENV_TRAIN_N_PROMPTS,
        "train_n_prompts": TRAIN_N_PROMPTS,
        "optimization_n_prompts": OPTIMIZATION_N_PROMPTS,
        "eval_n_prompts": EVAL_N_PROMPTS,
        "smoke_n_prompts": SMOKE_N_PROMPTS,
        "batch_size": BATCH_SIZE,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "global_k_values": GLOBAL_K_VALUES,
        "role_k_values": ROLE_K_VALUES,
        "top_global_feature_pool": TOP_GLOBAL_FEATURE_POOL,
        "global_soft_candidate_features": GLOBAL_SOFT_CANDIDATE_FEATURES,
        "run_global_soft_mask": RUN_GLOBAL_SOFT_MASK,
        "global_soft_lambdas": GLOBAL_SOFT_LAMBDAS,
        "global_soft_steps": GLOBAL_SOFT_STEPS,
        "global_soft_lr": GLOBAL_SOFT_LR,
        "global_soft_max_gain": GLOBAL_SOFT_MAX_GAIN,
        "global_attribution_prior_weight": GLOBAL_ATTRIBUTION_PRIOR_WEIGHT,
        "global_gain_anchor_weight": GLOBAL_GAIN_ANCHOR_WEIGHT,
        "answer_attribution_batch_size": ANSWER_ATTRIBUTION_BATCH_SIZE,
        "answer_attribution_prior_weight": None,
        "global_soft_top_k_values": GLOBAL_SOFT_TOP_K_VALUES,
        "global_soft_threshold_values": GLOBAL_SOFT_THRESHOLD_VALUES,
        "robust_worst_env_weight": ROBUST_WORST_ENV_WEIGHT,
        "robust_variance_weight": ROBUST_VARIANCE_WEIGHT,
        "robust_example_ratio_weight": ROBUST_EXAMPLE_RATIO_WEIGHT,
        "counterfactual_train_families": COUNTERFACTUAL_TRAIN_FAMILIES,
        "counterfactual_eval_families": COUNTERFACTUAL_EVAL_FAMILIES,
        "counterfactual_family_size": COUNTERFACTUAL_FAMILY_SIZE,
        "counterfactual_batch_families": COUNTERFACTUAL_BATCH_FAMILIES,
        "counterfactual_ratio_weight": COUNTERFACTUAL_RATIO_WEIGHT,
        "counterfactual_family_mean_weight": COUNTERFACTUAL_FAMILY_MEAN_WEIGHT,
        "counterfactual_name_variance_weight": COUNTERFACTUAL_NAME_VARIANCE_WEIGHT,
        "counterfactual_worst_family_weight": COUNTERFACTUAL_WORST_FAMILY_WEIGHT,
        "run_group_diagnostics": RUN_GROUP_DIAGNOSTICS,
        "run_counterfactual_diagnostics": RUN_COUNTERFACTUAL_DIAGNOSTICS,
        "group_diagnostic_top_n": GROUP_DIAGNOSTIC_TOP_N,
        "min_group_prompts": MIN_GROUP_PROMPTS,
        "faithfulness_band": [FAITHFULNESS_BAND_LOW, FAITHFULNESS_BAND_HIGH],
        "random_control_repeats": RANDOM_CONTROL_REPEATS,
        "random_control_k_values": RANDOM_CONTROL_K_VALUES,
        "stability_source_rankings": STABILITY_SOURCE_RANKINGS,
        "stability_pool_k": STABILITY_POOL_K,
        "stability_vote_top_k": STABILITY_VOTE_TOP_K,
        "stability_top_k_values": STABILITY_TOP_K_VALUES,
        "stability_summary_top_n": STABILITY_SUMMARY_TOP_N,
        "include_single_environment_rankings": INCLUDE_SINGLE_ENVIRONMENT_RANKINGS,
        "validation_global_baselines": VALIDATION_GLOBAL_BASELINES,
        "validation_split_seeds": VALIDATION_SPLIT_SEEDS,
        "optimization_env_names": OPTIMIZATION_ENV_NAMES,
        "final_validation_splits": FINAL_VALIDATION_SPLITS,
        "role_names": ROLE_NAMES,
        "core_role_names": CORE_ROLE_NAMES,
        "project_root": str(PROJECT_ROOT),
        "cache_dir": str(CACHE_DIR),
        "output_dir": str(OUTPUT_DIR),
        "latest_dir": str(LATEST_DIR),
        "artifacts": {
            "train_stats_cache_path": str(TRAIN_STATS_CACHE_PATH),
            "answer_direction_stats_cache_path": str(ANSWER_DIRECTION_STATS_CACHE_PATH),
            "smoke_results_csv_path": str(SMOKE_RESULTS_CSV_PATH),
            "results_csv_path": str(RESULTS_CSV_PATH),
            "summary_csv_path": str(SUMMARY_CSV_PATH),
            "invariant_stability_trace_csv_path": str(INVARIANT_TRACE_CSV_PATH),
            "masks_path": str(MASKS_PATH),
            "smoke_plot_path": str(SMOKE_PLOT_PATH),
            "pareto_plot_path": str(PARETO_PLOT_PATH),
            "heatmap_plot_path": str(HEATMAP_PLOT_PATH),
            "group_diagnostic_csv_path": str(GROUP_DIAGNOSTIC_CSV_PATH),
            "group_diagnostic_heatmap_path": str(GROUP_DIAGNOSTIC_HEATMAP_PATH),
            "counterfactual_diagnostic_csv_path": str(COUNTERFACTUAL_DIAGNOSTIC_CSV_PATH),
            "counterfactual_diagnostic_heatmap_path": str(COUNTERFACTUAL_DIAGNOSTIC_HEATMAP_PATH),
        },
        "extra": extra or {},
    }
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
    return manifest

def mirror_artifacts_to_latest(paths):
    LATEST_DIR.mkdir(parents=True, exist_ok=True)
    copied = []
    for artifact_path in paths:
        artifact_path = Path(artifact_path)
        if artifact_path.exists():
            destination = LATEST_DIR / artifact_path.name
            shutil.copy2(artifact_path, destination)
            copied.append(destination)
    return copied

write_run_manifest("initialized")
print(f"Project root: {PROJECT_ROOT}")
print(f"Run version: {RUN_VERSION}")
print(f"Trial ID: {TRIAL_ID}")
print(f"Output folder: {OUTPUT_DIR}")
print(f"Cache folder: {CACHE_DIR}")
print(f"Faithfulness band: {FAITHFULNESS_BAND_LOW} to {FAITHFULNESS_BAND_HIGH}")
print(f"Stability source rankings: {STABILITY_SOURCE_RANKINGS}")

## 3. Load Model and SAE

The v3 experiment uses the same layer 8 residual-stream SAE as v1/v2 so the curves are directly comparable.


In [ ]:
SAE_CACHE = {}

def load_sae_for_layer(layer):
    if layer in SAE_CACHE:
        return SAE_CACHE[layer]
    sae = SAE.from_pretrained(release=SAE_RELEASE, sae_id=sae_id_for_layer(layer), device=device)
    sae.eval()
    for param in sae.parameters():
        param.requires_grad_(False)
    assert sae.cfg.d_sae == EXPECTED_D_SAE
    SAE_CACHE[layer] = sae
    return sae

target_sae = load_sae_for_layer(TARGET_LAYER)
metadata = getattr(target_sae.cfg, "metadata", None)
model_kwargs = getattr(metadata, "model_from_pretrained_kwargs", None) or {}
model = HookedSAETransformer.from_pretrained_no_processing(MODEL_NAME, device=device, **model_kwargs)
model.eval()
for param in model.parameters():
    param.requires_grad_(False)

if getattr(model.tokenizer, "pad_token", None) is None:
    model.tokenizer.pad_token = model.tokenizer.eos_token
model.tokenizer.padding_side = "left"
if not getattr(model.tokenizer, "is_fast", False):
    raise RuntimeError("Role labeling requires a fast tokenizer with offset mappings.")

print(f"Loaded model: {MODEL_NAME}")
print(f"Loaded SAE: {SAE_RELEASE} / {sae_id_for_layer(TARGET_LAYER)}")
print(f"SAE d_in: {target_sae.cfg.d_in}")
print(f"SAE d_sae: {target_sae.cfg.d_sae}")


## 4. Build Name-Counterfactual IOI Datasets With Token Roles

V11 keeps the v8-v10 split discipline. Optimization environments are used to estimate circuit stability; final held-out names/templates are used only after masks have been built.

In [ ]:
BASE_NAMES = [
    "John", "Mary", "Bob", "Alice", "Tom", "Sarah", "James", "Emily",
    "Robert", "Laura", "Michael", "Anna", "David", "Lisa", "Daniel", "Emma",
    "Paul", "Karen", "Mark", "Susan", "Peter", "Linda", "Kevin", "Nancy",
    "Steven", "Helen", "George", "Carol", "Brian", "Julia", "Henry", "Megan",
    "Adam", "Rachel", "Patrick", "Olivia", "Andrew", "Grace", "Edward", "Sophie",
]
TRAIN_NAMES = BASE_NAMES[:24]
AUX_NAMES = BASE_NAMES[24:32]
HELDOUT_NAMES = BASE_NAMES[32:]
NAMES = TRAIN_NAMES + AUX_NAMES + HELDOUT_NAMES

PLACES = ["store", "park", "school", "office", "garden", "library", "station", "market", "museum", "theater", "church", "beach", "cafe", "hotel", "airport", "hospital"]
OBJECTS = ["book", "letter", "drink", "snack", "ticket", "phone", "gift", "photo", "bag", "card", "key", "toy", "coin", "map", "note", "pen"]

TRAIN_TEMPLATES = [
    "When {subject} and {io} went to the {place}, {subject} gave a {obj} to",
    "After {subject} and {io} visited the {place}, {subject} handed a {obj} to",
    "While {subject} and {io} waited near the {place}, {subject} passed a {obj} to",
    "Because {subject} and {io} were at the {place}, {subject} offered a {obj} to",
]
AUX_TEMPLATES = [
    "After {subject} and {io} arrived at the {place}, {subject} delivered the {obj} to",
    "Once {subject} and {io} met beside the {place}, {subject} sent the {obj} to",
    "Before {subject} and {io} left the {place}, {subject} showed the {obj} to",
    "Since {subject} and {io} stayed in the {place}, {subject} carried the {obj} to",
]
HELDOUT_TEMPLATES = [
    "After {subject} and {io} talked at the {place}, {subject} gave the {obj} to",
    "When {subject} and {io} paused inside the {place}, {subject} handed the {obj} to",
    "As {subject} and {io} walked through the {place}, {subject} passed the {obj} to",
    "Once {subject} and {io} gathered around the {place}, {subject} offered the {obj} to",
]
TEMPLATES = TRAIN_TEMPLATES + AUX_TEMPLATES + HELDOUT_TEMPLATES

assert set(TRAIN_NAMES).isdisjoint(AUX_NAMES)
assert set(TRAIN_NAMES).isdisjoint(HELDOUT_NAMES)
assert set(AUX_NAMES).isdisjoint(HELDOUT_NAMES)
assert set(TRAIN_TEMPLATES).isdisjoint(AUX_TEMPLATES)
assert set(TRAIN_TEMPLATES).isdisjoint(HELDOUT_TEMPLATES)
assert set(AUX_TEMPLATES).isdisjoint(HELDOUT_TEMPLATES)

def build_ioi_dataset(n_prompts=1_000, seed=0, names=None, places=None, objects=None, templates=None, exclude_clean_prompts=None):
    rng = random.Random(seed)
    name_pool = list(names or NAMES)
    place_pool = list(places or PLACES)
    object_pool = list(objects or OBJECTS)
    template_pool = list(templates or TEMPLATES)
    excluded = set(exclude_clean_prompts or [])
    records, seen, attempts = [], set(), 0
    while len(records) < n_prompts and attempts < n_prompts * 200:
        attempts += 1
        subject, io = rng.sample(name_pool, 2)
        place, obj, template = rng.choice(place_pool), rng.choice(object_pool), rng.choice(template_pool)
        clean_prompt = template.format(subject=subject, io=io, place=place, obj=obj)
        corrupt_prompt = template.format(subject=io, io=subject, place=place, obj=obj)
        key = (clean_prompt, corrupt_prompt)
        if key in seen or clean_prompt in excluded:
            continue
        seen.add(key)
        records.append({
            "clean_prompt": clean_prompt,
            "corrupt_prompt": corrupt_prompt,
            "answer_clean": f" {io}",
            "answer_corrupt": f" {subject}",
            "subject": subject,
            "indirect_object": io,
            "place": place,
            "object": obj,
            "template": template,
            "family_id": None,
            "family_member": None,
            "name_pair": f"{subject}->{io}",
        })
    if len(records) < n_prompts:
        raise ValueError(f"Only generated {len(records)} unique prompts out of requested {n_prompts}.")
    return pd.DataFrame(records)

def build_name_counterfactual_families(split_name, n_families, family_size, seed, names, templates, places=None, objects=None):
    rng = random.Random(seed)
    name_pool = list(names)
    template_pool = list(templates)
    place_pool = list(places or PLACES)
    object_pool = list(objects or OBJECTS)
    if len(name_pool) < family_size * 2:
        raise ValueError("Need enough names to build non-overlapping counterfactual family members.")
    records, seen_prompts, attempts = [], set(), 0
    family_idx = 0
    while family_idx < n_families and attempts < n_families * 200:
        attempts += 1
        template = rng.choice(template_pool)
        place = rng.choice(place_pool)
        obj = rng.choice(object_pool)
        family_id = f"{split_name}_{family_idx:04d}"
        member_pairs = []
        used_in_family = set()
        member_attempts = 0
        while len(member_pairs) < family_size and member_attempts < 200:
            member_attempts += 1
            subject, io = rng.sample(name_pool, 2)
            if subject in used_in_family or io in used_in_family:
                continue
            clean_prompt = template.format(subject=subject, io=io, place=place, obj=obj)
            if clean_prompt in seen_prompts:
                continue
            used_in_family.update([subject, io])
            seen_prompts.add(clean_prompt)
            member_pairs.append((subject, io, clean_prompt))
        if len(member_pairs) < family_size:
            continue
        for member_idx, (subject, io, clean_prompt) in enumerate(member_pairs):
            corrupt_prompt = template.format(subject=io, io=subject, place=place, obj=obj)
            records.append({
                "clean_prompt": clean_prompt,
                "corrupt_prompt": corrupt_prompt,
                "answer_clean": f" {io}",
                "answer_corrupt": f" {subject}",
                "subject": subject,
                "indirect_object": io,
                "place": place,
                "object": obj,
                "template": template,
                "family_id": family_id,
                "family_member": member_idx,
                "name_pair": f"{subject}->{io}",
            })
        family_idx += 1
    expected = n_families * family_size
    if len(records) < expected:
        raise ValueError(f"Only generated {len(records)} counterfactual rows out of requested {expected}.")
    return pd.DataFrame(records)

def token_id_for_answer(answer):
    tokens = model.to_tokens(answer, prepend_bos=False).reshape(-1)
    if tokens.numel() != 1:
        pieces = model.to_str_tokens(answer, prepend_bos=False)
        raise ValueError(f"Answer {answer!r} is not a single token: {pieces}")
    return int(tokens.item())

def find_required_span(text, value, start=0):
    index = text.find(value, start)
    if index < 0:
        raise ValueError(f"Could not find {value!r} in {text!r} after {start}")
    return index, index + len(value)

def overlap_len(a_start, a_end, b_start, b_end):
    return max(0, min(a_end, b_end) - max(a_start, b_start))

def role_ids_for_prompt(row):
    text = row["clean_prompt"]
    subject, io, place, obj = row["subject"], row["indirect_object"], row["place"], row["object"]
    subject_first = find_required_span(text, subject, 0)
    indirect_object = find_required_span(text, io, subject_first[1])
    subject_repeat = find_required_span(text, subject, indirect_object[1])
    place_span = find_required_span(text, place, indirect_object[1])
    object_span = find_required_span(text, obj, place_span[1])
    spans = [
        ("subject_first", subject_first),
        ("indirect_object", indirect_object),
        ("subject_repeat", subject_repeat),
        ("place", place_span),
        ("object", object_span),
    ]
    encoded = model.tokenizer(text, return_offsets_mapping=True, add_special_tokens=False)
    token_ids, offsets = encoded["input_ids"], encoded["offset_mapping"]
    tl_tokens = model.to_tokens(text, prepend_bos=True).squeeze(0)
    if len(token_ids) + 1 != int(tl_tokens.numel()):
        raise ValueError("Tokenizer offset length does not match TransformerLens token length with BOS.")
    roles = [ROLE_TO_ID["bos"]]
    for start, end in offsets:
        best_role, best_overlap = "other", 0
        for role_name, (span_start, span_end) in spans:
            current_overlap = overlap_len(start, end, span_start, span_end)
            if current_overlap > best_overlap:
                best_overlap, best_role = current_overlap, role_name
        roles.append(ROLE_TO_ID[best_role])
    roles[-1] = ROLE_TO_ID["final_prediction"]
    return roles

def add_answer_tokens_and_roles(dataset, split_name):
    dataset = dataset.copy()
    dataset["split"] = split_name
    dataset["answer_clean_id"] = [token_id_for_answer(x) for x in dataset["answer_clean"]]
    dataset["answer_corrupt_id"] = [token_id_for_answer(x) for x in dataset["answer_corrupt"]]
    dataset["role_ids"] = [role_ids_for_prompt(row) for _, row in dataset.iterrows()]
    return dataset.reset_index(drop=True)

def make_dataset_split(split_name, n_prompts, seed, names, templates, exclude_clean_prompts=None):
    raw = build_ioi_dataset(
        n_prompts=n_prompts,
        seed=seed,
        names=names,
        templates=templates,
        exclude_clean_prompts=exclude_clean_prompts,
    )
    return add_answer_tokens_and_roles(raw, split_name=split_name)

def make_counterfactual_family_split(split_name, n_families, family_size, seed, names, templates):
    raw = build_name_counterfactual_families(
        split_name=split_name,
        n_families=n_families,
        family_size=family_size,
        seed=seed,
        names=names,
        templates=templates,
    )
    return add_answer_tokens_and_roles(raw, split_name=split_name)

smoke_dataset = make_dataset_split("smoke", SMOKE_N_PROMPTS, SEED, TRAIN_NAMES, TRAIN_TEMPLATES)
train_core_dataset = make_dataset_split("train_core", ENV_TRAIN_N_PROMPTS, VALIDATION_SPLIT_SEEDS["train_core"], TRAIN_NAMES, TRAIN_TEMPLATES)
train_template_aug_dataset = make_dataset_split("train_template_aug", ENV_TRAIN_N_PROMPTS, VALIDATION_SPLIT_SEEDS["train_template_aug"], TRAIN_NAMES, AUX_TEMPLATES)
train_name_aug_dataset = make_dataset_split("train_name_aug", ENV_TRAIN_N_PROMPTS, VALIDATION_SPLIT_SEEDS["train_name_aug"], AUX_NAMES, TRAIN_TEMPLATES)
train_crossed_aug_dataset = make_dataset_split("train_crossed_aug", ENV_TRAIN_N_PROMPTS, VALIDATION_SPLIT_SEEDS["train_crossed_aug"], AUX_NAMES, AUX_TEMPLATES)
name_cf_train_dataset = make_counterfactual_family_split(
    "name_cf_train",
    COUNTERFACTUAL_TRAIN_FAMILIES,
    COUNTERFACTUAL_FAMILY_SIZE,
    VALIDATION_SPLIT_SEEDS["name_cf_train"],
    TRAIN_NAMES + AUX_NAMES,
    TRAIN_TEMPLATES + AUX_TEMPLATES,
)
optimization_env_datasets = {
    "train_core": train_core_dataset,
    "train_template_aug": train_template_aug_dataset,
    "train_name_aug": train_name_aug_dataset,
    "train_crossed_aug": train_crossed_aug_dataset,
    "name_cf_train": name_cf_train_dataset,
}
train_dataset = pd.concat(list(optimization_env_datasets.values()), ignore_index=True)
optimization_dataset = train_core_dataset.copy()
train_prompt_set = set(train_dataset["clean_prompt"])

id_test_dataset = make_dataset_split("id_test", EVAL_N_PROMPTS, VALIDATION_SPLIT_SEEDS["id_test"], TRAIN_NAMES, TRAIN_TEMPLATES, exclude_clean_prompts=train_prompt_set)
heldout_template_dataset = make_dataset_split("heldout_templates", EVAL_N_PROMPTS, VALIDATION_SPLIT_SEEDS["heldout_templates"], TRAIN_NAMES, HELDOUT_TEMPLATES)
heldout_name_dataset = make_dataset_split("heldout_names", EVAL_N_PROMPTS, VALIDATION_SPLIT_SEEDS["heldout_names"], HELDOUT_NAMES, TRAIN_TEMPLATES)
heldout_template_name_dataset = make_dataset_split("heldout_templates_names", EVAL_N_PROMPTS, VALIDATION_SPLIT_SEEDS["heldout_templates_names"], HELDOUT_NAMES, HELDOUT_TEMPLATES)

heldout_name_family_dataset = make_counterfactual_family_split(
    "heldout_name_families",
    COUNTERFACTUAL_EVAL_FAMILIES,
    COUNTERFACTUAL_FAMILY_SIZE,
    VALIDATION_SPLIT_SEEDS["heldout_name_families"],
    HELDOUT_NAMES,
    TRAIN_TEMPLATES,
)
heldout_template_name_family_dataset = make_counterfactual_family_split(
    "heldout_template_name_families",
    COUNTERFACTUAL_EVAL_FAMILIES,
    COUNTERFACTUAL_FAMILY_SIZE,
    VALIDATION_SPLIT_SEEDS["heldout_template_name_families"],
    HELDOUT_NAMES,
    HELDOUT_TEMPLATES,
)

validation_datasets = {
    "train_core": train_core_dataset,
    "train_template_aug": train_template_aug_dataset,
    "train_name_aug": train_name_aug_dataset,
    "train_crossed_aug": train_crossed_aug_dataset,
    "name_cf_train": name_cf_train_dataset,
    "id_test": id_test_dataset,
    "heldout_templates": heldout_template_dataset,
    "heldout_names": heldout_name_dataset,
    "heldout_templates_names": heldout_template_name_dataset,
}

counterfactual_family_datasets = {
    "name_cf_train": name_cf_train_dataset,
    "heldout_name_families": heldout_name_family_dataset,
    "heldout_template_name_families": heldout_template_name_family_dataset,
}

for split_name, split_df in validation_datasets.items():
    overlap = len(train_prompt_set.intersection(set(split_df["clean_prompt"])))
    final_tag = "final" if split_name in FINAL_VALIDATION_SPLITS else "optimization"
    n_families = split_df["family_id"].dropna().nunique() if "family_id" in split_df else 0
    family_note = f", families={n_families}" if n_families else ""
    print(f"{split_name:>24}: {len(split_df):4d} prompts, train prompt overlap={overlap}, {final_tag}{family_note}")

for split_name, split_df in counterfactual_family_datasets.items():
    print(f"{split_name:>32}: {split_df['family_id'].nunique():4d} counterfactual families, {len(split_df):4d} prompts")

print(smoke_dataset[["clean_prompt", "answer_clean", "answer_corrupt"]].head())
print(f"Optimization environments: {list(optimization_env_datasets)}")
print(f"Pooled train prompts for stats: {len(train_dataset)}")
print("Example roles:", [ROLE_NAMES[x] for x in smoke_dataset.loc[0, "role_ids"]])

## 5. Role-Conditioned Intervention Helpers

The hook supports both v2-style global masks and v3 role-feature masks. `preserve_error=True` is kept from v2 so all-feature masks recover the full activation exactly.


In [ ]:
def group_tokenized_rows(dataset):
    groups = defaultdict(list)
    for index, row in dataset.reset_index(drop=True).iterrows():
        tokens = model.to_tokens(row["clean_prompt"], prepend_bos=True).squeeze(0).to(device)
        role_ids = torch.tensor(row["role_ids"], dtype=torch.long, device=device)
        if int(tokens.numel()) != int(role_ids.numel()):
            raise ValueError("Token and role lengths do not match.")
        groups[int(tokens.numel())].append((index, tokens, role_ids))
    return groups

def sae_role_mask_hook(activation, hook, sae, feature_mask=None, role_feature_mask=None, role_ids=None, preserve_error=True):
    feature_acts = sae.encode(activation)
    full_reconstruction = sae.decode(feature_acts)
    if role_feature_mask is not None:
        if role_ids is None:
            raise ValueError("role_ids are required for role_feature_mask")
        gate = role_feature_mask.to(device=feature_acts.device, dtype=feature_acts.dtype)[role_ids]
        masked_feature_acts = feature_acts * gate
    elif feature_mask is not None:
        gate = feature_mask.to(device=feature_acts.device, dtype=feature_acts.dtype).view(1, 1, -1)
        masked_feature_acts = feature_acts * gate
    else:
        masked_feature_acts = feature_acts
    masked_reconstruction = sae.decode(masked_feature_acts)
    if preserve_error:
        return masked_reconstruction + (activation - full_reconstruction)
    return masked_reconstruction

def run_logits(dataset, batch_size=BATCH_SIZE, sae=None, feature_mask=None, role_feature_mask=None, use_sae_substitution=False, preserve_error=True, detach=True):
    output_logits = [None] * len(dataset)
    groups = group_tokenized_rows(dataset)
    hook_name = hook_name_for_layer(TARGET_LAYER)
    grad_context = torch.no_grad() if detach else torch.enable_grad()
    with grad_context:
        for items in groups.values():
            for start in range(0, len(items), batch_size):
                chunk = items[start : start + batch_size]
                indices = [item[0] for item in chunk]
                tokens = torch.stack([item[1] for item in chunk], dim=0)
                role_ids = torch.stack([item[2] for item in chunk], dim=0)
                if use_sae_substitution:
                    fwd_hooks = [(hook_name, partial(sae_role_mask_hook, sae=sae, feature_mask=feature_mask, role_feature_mask=role_feature_mask, role_ids=role_ids, preserve_error=preserve_error))]
                    context = model.hooks(fwd_hooks=fwd_hooks)
                else:
                    context = nullcontext()
                with context:
                    final_logits = model(tokens)[:, -1, :]
                if detach:
                    final_logits = final_logits.detach().cpu()
                for idx, row_logits in zip(indices, final_logits):
                    output_logits[idx] = row_logits
    return torch.stack(output_logits, dim=0)

def example_logit_diffs(final_logits, dataset):
    clean_ids = torch.tensor(dataset["answer_clean_id"].to_list(), dtype=torch.long, device=final_logits.device)
    corrupt_ids = torch.tensor(dataset["answer_corrupt_id"].to_list(), dtype=torch.long, device=final_logits.device)
    row_indices = torch.arange(len(dataset), dtype=torch.long, device=final_logits.device)
    return final_logits[row_indices, clean_ids] - final_logits[row_indices, corrupt_ids]

def mean_logit_diff(final_logits, dataset):
    return example_logit_diffs(final_logits, dataset).mean()

def compute_faithfulness(sae, dataset, feature_mask=None, role_feature_mask=None, full_logit_diff=None, batch_size=BATCH_SIZE):
    if full_logit_diff is None:
        full_logits = run_logits(dataset, batch_size=batch_size, detach=True)
        full_logit_diff = mean_logit_diff(full_logits, dataset)
    masked_logits = run_logits(dataset, batch_size=batch_size, sae=sae, feature_mask=feature_mask, role_feature_mask=role_feature_mask, use_sae_substitution=True, preserve_error=True, detach=True)
    masked_logit_diff = mean_logit_diff(masked_logits, dataset)
    denominator = float(full_logit_diff.item())
    if abs(denominator) < 1e-8:
        raise ValueError(f"Full-model logit diff is too close to zero: {denominator}")
    if role_feature_mask is not None:
        active_nodes = int((role_feature_mask > 0).sum().item())
        unique_features = int(((role_feature_mask > 0).sum(dim=0) > 0).sum().item())
        mask_type = "role_feature"
    elif feature_mask is not None:
        active_nodes = int((feature_mask > 0).sum().item())
        unique_features = active_nodes
        mask_type = "global_feature"
    else:
        active_nodes = int(sae.cfg.d_sae)
        unique_features = active_nodes
        mask_type = "all_features"
    return {
        "faithfulness": float((masked_logit_diff / full_logit_diff).item()),
        "full_logit_diff": float(full_logit_diff.item()),
        "masked_logit_diff": float(masked_logit_diff.item()),
        "active_nodes": active_nodes,
        "unique_features": unique_features,
        "mask_type": mask_type,
    }

def make_global_feature_mask(top_features, k, d_sae=EXPECTED_D_SAE):
    k = min(int(k), int(d_sae))
    mask = torch.zeros(int(d_sae), device=device)
    mask[top_features[:k].to(device)] = 1.0
    return mask

def make_role_feature_mask(pair_ranking, k, n_roles=len(ROLE_NAMES), d_sae=EXPECTED_D_SAE):
    k = min(int(k), int(pair_ranking.numel()))
    flat_ids = pair_ranking[:k].to(device)
    roles = torch.div(flat_ids, d_sae, rounding_mode="floor")
    features = flat_ids % d_sae
    mask = torch.zeros((n_roles, d_sae), device=device)
    mask[roles, features] = 1.0
    return mask


## 6. Dimension and Role Check

In [ ]:
sample_tokens = model.to_tokens(smoke_dataset.loc[0, "clean_prompt"], prepend_bos=True).to(device)
_, sample_cache = model.run_with_cache(sample_tokens, names_filter=[hook_name_for_layer(TARGET_LAYER)])
sample_acts = sample_cache[hook_name_for_layer(TARGET_LAYER)]
print(f"Sample activation shape at layer {TARGET_LAYER}: {tuple(sample_acts.shape)}")
print(f"SAE d_in: {target_sae.cfg.d_in}")
print(f"SAE d_sae: {target_sae.cfg.d_sae}")
assert sample_acts.shape[-1] == target_sae.cfg.d_in
assert target_sae.cfg.d_sae == EXPECTED_D_SAE
for _, row in smoke_dataset.iterrows():
    token_count = int(model.to_tokens(row["clean_prompt"], prepend_bos=True).numel())
    assert token_count == len(row["role_ids"])
print("Role labels align with tokenized prompts.")


## 7. Cache Global and Role-Feature Statistics

In [ ]:
def cache_role_feature_stats(dataset, sae, cache_path, batch_size=BATCH_SIZE, force_recompute=False):
    cache_path = Path(cache_path)
    if cache_path.exists() and not force_recompute:
        payload = torch.load(cache_path, map_location="cpu")
        print(f"Loaded cached role-feature stats from {cache_path}")
        return {key: value.to(device) if torch.is_tensor(value) else value for key, value in payload.items()}
    d_sae, n_roles = int(sae.cfg.d_sae), len(ROLE_NAMES)
    sum_abs_all = torch.zeros(d_sae, device=device)
    sum_sq_all = torch.zeros(d_sae, device=device)
    count_all = 0
    sum_abs_by_role = torch.zeros((n_roles, d_sae), device=device)
    sum_sq_by_role = torch.zeros((n_roles, d_sae), device=device)
    count_by_role = torch.zeros(n_roles, device=device)
    groups = group_tokenized_rows(dataset)
    hook_name = hook_name_for_layer(TARGET_LAYER)
    with torch.no_grad():
        for items in tqdm(groups.values(), desc="Token length groups"):
            for start in tqdm(range(0, len(items), batch_size), leave=False, desc="Batches"):
                chunk = items[start : start + batch_size]
                tokens = torch.stack([item[1] for item in chunk], dim=0)
                role_ids = torch.stack([item[2] for item in chunk], dim=0)
                _, cache = model.run_with_cache(tokens, names_filter=[hook_name])
                feature_acts = sae.encode(cache[hook_name])
                sum_abs_all += feature_acts.abs().sum(dim=(0, 1))
                sum_sq_all += feature_acts.square().sum(dim=(0, 1))
                count_all += feature_acts.shape[0] * feature_acts.shape[1]
                for role_id in range(n_roles):
                    selected = feature_acts[role_ids == role_id]
                    if selected.numel() == 0:
                        continue
                    sum_abs_by_role[role_id] += selected.abs().sum(dim=0)
                    sum_sq_by_role[role_id] += selected.square().sum(dim=0)
                    count_by_role[role_id] += selected.shape[0]
    safe_role_counts = count_by_role.clamp_min(1).view(-1, 1)
    payload = {
        "mean_abs_all": (sum_abs_all / count_all).detach().cpu(),
        "l2_all": sum_sq_all.sqrt().detach().cpu(),
        "mean_abs_by_role": (sum_abs_by_role / safe_role_counts).detach().cpu(),
        "l2_by_role": sum_sq_by_role.sqrt().detach().cpu(),
        "count_all": int(count_all),
        "count_by_role": count_by_role.detach().cpu(),
        "role_names": ROLE_NAMES,
        "target_layer": TARGET_LAYER,
        "sae_release": SAE_RELEASE,
        "sae_id": sae_id_for_layer(TARGET_LAYER),
    }
    torch.save(payload, cache_path)
    print(f"Saved role-feature stats to {cache_path}")
    return {key: value.to(device) if torch.is_tensor(value) else value for key, value in payload.items()}

def get_decoder_feature_norm(sae):
    W_dec = sae.W_dec.detach()
    if W_dec.shape[0] == int(sae.cfg.d_sae):
        return W_dec.norm(dim=1)
    if W_dec.shape[-1] == int(sae.cfg.d_sae):
        return W_dec.norm(dim=0)
    raise ValueError(f"Cannot infer decoder feature axis from W_dec shape {tuple(W_dec.shape)}")

def build_pair_ranking(score_by_role, allowed_features=None, allowed_role_ids=None):
    scores = score_by_role.clone().to(device)
    if allowed_features is not None:
        allowed_mask = torch.zeros(scores.shape[1], dtype=torch.bool, device=device)
        allowed_mask[allowed_features.to(device)] = True
        scores[:, ~allowed_mask] = -torch.inf
    if allowed_role_ids is not None:
        role_mask = torch.zeros(scores.shape[0], dtype=torch.bool, device=device)
        role_mask[allowed_role_ids.to(device)] = True
        scores[~role_mask, :] = -torch.inf
    return scores.flatten().argsort(descending=True)

def make_rankings(stats, sae):
    decoder_norm = get_decoder_feature_norm(sae).to(device)
    global_score = stats["mean_abs_all"]
    global_ranking = global_score.argsort(descending=True)
    allowed_features = global_ranking[:TOP_GLOBAL_FEATURE_POOL]
    role_mean_abs = stats["mean_abs_by_role"]
    role_wanda = stats["l2_by_role"] * decoder_norm.view(1, -1)
    core_role_ids = CORE_ROLE_IDS.to(device)
    rankings = {
        "global_activation_mean_abs": global_ranking,
        "role_mean_abs_all_roles": build_pair_ranking(role_mean_abs, allowed_features=allowed_features),
        "role_mean_abs_core_roles": build_pair_ranking(role_mean_abs, allowed_features=allowed_features, allowed_role_ids=core_role_ids),
        "role_wanda_all_roles": build_pair_ranking(role_wanda, allowed_features=allowed_features),
        "role_wanda_core_roles": build_pair_ranking(role_wanda, allowed_features=allowed_features, allowed_role_ids=core_role_ids),
    }
    return {"global_score": global_score, "role_mean_abs": role_mean_abs, "role_wanda": role_wanda}, rankings



def decoder_feature_matrix(sae):
    W_dec = sae.W_dec.detach()
    d_sae = int(sae.cfg.d_sae)
    if W_dec.shape[0] == d_sae:
        return W_dec
    if W_dec.shape[-1] == d_sae:
        return W_dec.T
    raise ValueError(f"Cannot infer decoder feature axis from W_dec shape {tuple(W_dec.shape)}")

def cache_answer_direction_feature_stats(dataset, sae, cache_path, batch_size=ANSWER_ATTRIBUTION_BATCH_SIZE, force_recompute=False):
    cache_path = Path(cache_path)
    if cache_path.exists() and not force_recompute:
        payload = torch.load(cache_path, map_location="cpu")
        print(f"Loaded cached answer-direction feature stats from {cache_path}")
        return {key: value.to(device) if torch.is_tensor(value) else value for key, value in payload.items()}

    d_sae, n_roles = int(sae.cfg.d_sae), len(ROLE_NAMES)
    sum_abs_all = torch.zeros(d_sae, device=device)
    sum_positive_all = torch.zeros(d_sae, device=device)
    sum_signed_all = torch.zeros(d_sae, device=device)
    sum_sq_all = torch.zeros(d_sae, device=device)
    count_all = 0

    sum_abs_by_role = torch.zeros((n_roles, d_sae), device=device)
    sum_positive_by_role = torch.zeros((n_roles, d_sae), device=device)
    sum_signed_by_role = torch.zeros((n_roles, d_sae), device=device)
    sum_sq_by_role = torch.zeros((n_roles, d_sae), device=device)
    count_by_role = torch.zeros(n_roles, device=device)

    groups = group_tokenized_rows(dataset)
    hook_name = hook_name_for_layer(TARGET_LAYER)
    decoder_matrix = decoder_feature_matrix(sae).to(device)
    batch_counter = 0
    full_diff_values = []
    direction_norm_values = []

    for items in tqdm(groups.values(), desc="Answer-direction token length groups"):
        for start in tqdm(range(0, len(items), batch_size), leave=False, desc="Attribution batches"):
            chunk = items[start : start + batch_size]
            indices = [item[0] for item in chunk]
            tokens = torch.stack([item[1] for item in chunk], dim=0)
            role_ids = torch.stack([item[2] for item in chunk], dim=0)
            clean_ids = torch.tensor(dataset.iloc[indices]["answer_clean_id"].to_list(), dtype=torch.long, device=device)
            corrupt_ids = torch.tensor(dataset.iloc[indices]["answer_corrupt_id"].to_list(), dtype=torch.long, device=device)
            saved = {}

            def capture_layer8_activation(activation, hook):
                leaf_activation = activation.detach().requires_grad_(True)
                leaf_activation.retain_grad()
                saved["activation"] = leaf_activation
                return leaf_activation

            model.zero_grad(set_to_none=True)
            with torch.enable_grad():
                with model.hooks(fwd_hooks=[(hook_name, capture_layer8_activation)]):
                    logits = model(tokens)[:, -1, :]
                    row_ids = torch.arange(tokens.shape[0], device=device)
                    diffs = logits[row_ids, clean_ids] - logits[row_ids, corrupt_ids]
                    answer_dirs = model.W_U[:, clean_ids].T - model.W_U[:, corrupt_ids].T
                    direction_norms = answer_dirs.norm(dim=-1).detach().clamp_min(ANSWER_ATTRIBUTION_SCORE_EPS)
                    scale = diffs.detach().abs().clamp_min(0.05) * direction_norms
                    objective = (diffs / scale).sum()
                objective.backward()

            activation = saved.get("activation")
            if activation is None or activation.grad is None:
                raise RuntimeError("Could not capture layer activation gradients for answer-direction attribution.")
            grads = activation.grad.detach()
            with torch.no_grad():
                feature_acts = sae.encode(activation.detach())
                projected_grads = torch.einsum("btd,fd->btf", grads, decoder_matrix)
                contributions = feature_acts * projected_grads

                sum_abs_all += contributions.abs().sum(dim=(0, 1))
                sum_positive_all += contributions.clamp_min(0).sum(dim=(0, 1))
                sum_signed_all += contributions.sum(dim=(0, 1))
                sum_sq_all += contributions.square().sum(dim=(0, 1))
                count_all += int(contributions.shape[0] * contributions.shape[1])

                for role_id in range(n_roles):
                    selected = contributions[role_ids == role_id]
                    if selected.numel() == 0:
                        continue
                    sum_abs_by_role[role_id] += selected.abs().sum(dim=0)
                    sum_positive_by_role[role_id] += selected.clamp_min(0).sum(dim=0)
                    sum_signed_by_role[role_id] += selected.sum(dim=0)
                    sum_sq_by_role[role_id] += selected.square().sum(dim=0)
                    count_by_role[role_id] += selected.shape[0]

                full_diff_values.extend(diffs.detach().cpu().tolist())
                direction_norm_values.extend(direction_norms.detach().cpu().tolist())

            batch_counter += 1
            if batch_counter % 10 == 0:
                torch.cuda.empty_cache()

    safe_role_counts = count_by_role.clamp_min(1).view(-1, 1)
    mean_abs_by_role = sum_abs_by_role / safe_role_counts
    mean_positive_by_role = sum_positive_by_role / safe_role_counts
    mean_signed_by_role = sum_signed_by_role / safe_role_counts
    rms_by_role = (sum_sq_by_role / safe_role_counts).sqrt()
    consistency_by_role = mean_signed_by_role / mean_abs_by_role.clamp_min(ANSWER_ATTRIBUTION_SCORE_EPS)
    support_by_role = mean_signed_by_role.clamp_min(0) * consistency_by_role.clamp_min(0)

    mean_abs_all = sum_abs_all / max(1, count_all)
    mean_positive_all = sum_positive_all / max(1, count_all)
    mean_signed_all = sum_signed_all / max(1, count_all)
    rms_all = (sum_sq_all / max(1, count_all)).sqrt()
    consistency_all = mean_signed_all / mean_abs_all.clamp_min(ANSWER_ATTRIBUTION_SCORE_EPS)
    support_all = mean_signed_all.clamp_min(0) * consistency_all.clamp_min(0)

    payload = {
        "mean_abs_all": mean_abs_all.detach().cpu(),
        "mean_positive_all": mean_positive_all.detach().cpu(),
        "mean_signed_all": mean_signed_all.detach().cpu(),
        "rms_all": rms_all.detach().cpu(),
        "consistency_all": consistency_all.detach().cpu(),
        "support_all": support_all.detach().cpu(),
        "mean_abs_by_role": mean_abs_by_role.detach().cpu(),
        "mean_positive_by_role": mean_positive_by_role.detach().cpu(),
        "mean_signed_by_role": mean_signed_by_role.detach().cpu(),
        "rms_by_role": rms_by_role.detach().cpu(),
        "consistency_by_role": consistency_by_role.detach().cpu(),
        "support_by_role": support_by_role.detach().cpu(),
        "count_all": int(count_all),
        "count_by_role": count_by_role.detach().cpu(),
        "mean_full_logit_diff": float(pd.Series(full_diff_values).mean()),
        "mean_answer_direction_norm": float(pd.Series(direction_norm_values).mean()),
        "role_names": ROLE_NAMES,
        "target_layer": TARGET_LAYER,
        "sae_release": SAE_RELEASE,
        "sae_id": sae_id_for_layer(TARGET_LAYER),
        "normalization": "logit_diff_abs_times_answer_unembedding_direction_norm",
    }
    torch.save(payload, cache_path)
    print(f"Saved answer-direction feature stats to {cache_path}")
    return {key: value.to(device) if torch.is_tensor(value) else value for key, value in payload.items()}

def make_answer_direction_rankings(answer_stats, sae):
    role_abs = answer_stats["mean_abs_by_role"].to(device)
    role_positive = answer_stats["mean_positive_by_role"].to(device)
    role_signed = answer_stats["mean_signed_by_role"].to(device)
    role_support = answer_stats["support_by_role"].to(device)
    role_consistency = answer_stats["consistency_by_role"].to(device).clamp_min(0)
    global_abs = answer_stats["mean_abs_all"].to(device)
    global_support = answer_stats["support_all"].to(device)
    allowed_features = global_support.argsort(descending=True)[:TOP_GLOBAL_FEATURE_POOL]
    core_role_ids = CORE_ROLE_IDS.to(device)
    support_consistency = role_support * role_consistency
    rankings = {
        "global_answer_gradient_abs": global_abs.argsort(descending=True),
        "global_answer_gradient_support": global_support.argsort(descending=True),
        "answer_gradient_abs_all_roles": build_pair_ranking(role_abs, allowed_features=allowed_features),
        "answer_gradient_positive_all_roles": build_pair_ranking(role_positive, allowed_features=allowed_features),
        "answer_gradient_support_all_roles": build_pair_ranking(role_support, allowed_features=allowed_features),
        "answer_gradient_support_core_roles": build_pair_ranking(role_support, allowed_features=allowed_features, allowed_role_ids=core_role_ids),
        "answer_gradient_consistent_core_roles": build_pair_ranking(support_consistency, allowed_features=allowed_features, allowed_role_ids=core_role_ids),
    }
    scores = {
        "answer_gradient_abs_by_role": role_abs,
        "answer_gradient_positive_by_role": role_positive,
        "answer_gradient_signed_by_role": role_signed,
        "answer_gradient_support_by_role": role_support,
        "answer_gradient_consistency_by_role": role_consistency,
        "answer_gradient_global_abs": global_abs,
        "answer_gradient_global_support": global_support,
    }
    return scores, rankings

def describe_role_feature_pairs(pair_ids, limit=10, d_sae=EXPECTED_D_SAE):
    rows = []
    for pair_id in pair_ids[:limit].detach().cpu().tolist():
        role_id = int(pair_id) // d_sae
        feature_id = int(pair_id) % d_sae
        rows.append({"role": ROLE_NAMES[role_id], "feature": feature_id, "pair_id": int(pair_id)})
    return pd.DataFrame(rows)


def positive_normalize(score, eps=ANSWER_ATTRIBUTION_SCORE_EPS):
    score = torch.nan_to_num(score.to(device=device, dtype=torch.float32), nan=0.0, posinf=0.0, neginf=0.0).clamp_min(0)
    positive = score[score > 0]
    if positive.numel() == 0:
        return score
    scale = torch.quantile(positive, 0.99).clamp_min(eps)
    return (score / scale).clamp(0, 5)

def rank_to_borda_score(ranking, d_sae=EXPECTED_D_SAE):
    ranking = ranking.to(device)
    values = torch.linspace(1.0, 0.0, steps=int(d_sae), device=device)
    score = torch.zeros(int(d_sae), device=device)
    score[ranking[: int(d_sae)]] = values[: ranking[: int(d_sae)].numel()]
    return score

def make_global_calibrated_rankings(train_stats, answer_stats, existing_rankings):
    activation = positive_normalize(train_stats["mean_abs_all"])
    activation_l2 = positive_normalize(train_stats["l2_all"])
    answer_abs = positive_normalize(answer_stats["mean_abs_all"])
    answer_positive = positive_normalize(answer_stats["mean_positive_all"])
    answer_support = positive_normalize(answer_stats["support_all"])
    answer_consistency = positive_normalize(answer_stats["consistency_all"].clamp_min(0))

    activation_borda = rank_to_borda_score(existing_rankings["global_activation_mean_abs"])
    answer_abs_borda = rank_to_borda_score(existing_rankings["global_answer_gradient_abs"])
    answer_support_borda = rank_to_borda_score(existing_rankings["global_answer_gradient_support"])

    scores = {
        "global_calibrated_activation_answer_abs_product": activation * answer_abs,
        "global_calibrated_activation_answer_support_product": activation * answer_support,
        "global_calibrated_answer_support_consistent": answer_support * (0.25 + answer_consistency),
        "global_calibrated_activation_l2_answer_abs": activation_l2 * answer_abs,
        "global_calibrated_borda_activation_answer": 0.45 * activation_borda + 0.35 * answer_abs_borda + 0.20 * answer_support_borda,
        "global_calibrated_rescue_union": 0.40 * activation + 0.35 * answer_abs + 0.20 * answer_support + 0.05 * answer_positive,
    }
    rankings = {name: score.argsort(descending=True) for name, score in scores.items()}
    return scores, rankings

def describe_global_features(feature_ids, score_table=None, limit=12):
    rows = []
    for feature_id in feature_ids[:limit].detach().cpu().tolist():
        row = {"feature": int(feature_id)}
        if score_table:
            for name, score in score_table.items():
                if torch.is_tensor(score) and score.ndim == 1:
                    row[name] = float(score[int(feature_id)].detach().cpu().item())
        rows.append(row)
    return pd.DataFrame(rows)

def make_soft_global_feature_mask(feature_ranking, feature_gains, top_k=None, threshold=None, d_sae=EXPECTED_D_SAE):
    feature_ranking = feature_ranking.to(device)
    feature_gains = feature_gains.to(device)
    keep = torch.ones(feature_gains.shape, dtype=torch.bool, device=device)
    if top_k is not None:
        keep &= torch.arange(feature_gains.numel(), device=device) < min(int(top_k), int(feature_gains.numel()))
    if threshold is not None:
        keep &= feature_gains >= float(threshold)
    selected_features = feature_ranking[keep]
    selected_gains = feature_gains[keep]
    mask = torch.zeros(int(d_sae), device=device, dtype=feature_gains.dtype)
    if selected_features.numel() > 0:
        mask[selected_features] = selected_gains
    return mask

def dense_global_mask_from_candidate_values(candidate_features, values, d_sae=EXPECTED_D_SAE):
    dense = torch.zeros(int(d_sae), device=values.device, dtype=values.dtype)
    dense = dense.scatter(0, candidate_features.to(values.device), values)
    return dense

## 8. Pareto Evaluation Helpers

In [ ]:
def evaluate_ranked_masks(dataset, sae, rankings, global_k_values, role_k_values, batch_size=BATCH_SIZE, output_csv_path=None, soft_gate_specs=None, soft_top_k_values=None, soft_threshold_values=None):
    full_logits = run_logits(dataset, batch_size=batch_size, detach=True)
    full_logit_diff = mean_logit_diff(full_logits, dataset)
    rows = []
    all_feature_mask = torch.ones(int(sae.cfg.d_sae), device=device)
    zero_feature_mask = torch.zeros(int(sae.cfg.d_sae), device=device)
    for baseline_name, feature_mask in [("all_features", all_feature_mask), ("zero_features", zero_feature_mask)]:
        metrics = compute_faithfulness(sae, dataset, feature_mask=feature_mask, full_logit_diff=full_logit_diff, batch_size=batch_size)
        rows.append({"baseline": baseline_name, "selection": "reference", "k": int(metrics["active_nodes"]), "threshold": None, "gate_sum": float(metrics["active_nodes"]), **metrics})
    for baseline_name, ranking in rankings.items():
        if baseline_name.startswith("global_"):
            for k in tqdm(global_k_values, desc=f"{baseline_name} sweep"):
                feature_mask = make_global_feature_mask(ranking, k, d_sae=int(sae.cfg.d_sae))
                metrics = compute_faithfulness(sae, dataset, feature_mask=feature_mask, full_logit_diff=full_logit_diff, batch_size=batch_size)
                rows.append({"baseline": baseline_name, "selection": "hard_topk", "k": int(k), "threshold": None, "gate_sum": float(metrics["active_nodes"]), **metrics})
        else:
            for k in tqdm(role_k_values, desc=f"{baseline_name} sweep"):
                role_feature_mask = make_role_feature_mask(ranking, k, n_roles=len(ROLE_NAMES), d_sae=int(sae.cfg.d_sae))
                metrics = compute_faithfulness(sae, dataset, role_feature_mask=role_feature_mask, full_logit_diff=full_logit_diff, batch_size=batch_size)
                rows.append({"baseline": baseline_name, "selection": "hard_topk", "k": int(k), "threshold": None, "gate_sum": float(metrics["active_nodes"]), **metrics})
    if soft_gate_specs:
        soft_top_k_values = soft_top_k_values or []
        soft_threshold_values = soft_threshold_values or []
        for baseline_name, spec in soft_gate_specs.items():
            pair_ranking = spec["pair_ranking"]
            pair_probs = spec["pair_probs"]
            for k in tqdm(soft_top_k_values, desc=f"{baseline_name} soft top-k"):
                role_feature_mask = make_soft_role_feature_mask(pair_ranking, pair_probs, top_k=int(k), n_roles=len(ROLE_NAMES), d_sae=int(sae.cfg.d_sae))
                metrics = compute_faithfulness(sae, dataset, role_feature_mask=role_feature_mask, full_logit_diff=full_logit_diff, batch_size=batch_size)
                rows.append({"baseline": baseline_name, "selection": "soft_topk", "k": int(k), "threshold": None, "gate_sum": float(role_feature_mask.sum().item()), "mean_nonzero_gate": mean_nonzero_gate(role_feature_mask), **metrics})
            for threshold in tqdm(soft_threshold_values, desc=f"{baseline_name} threshold"):
                role_feature_mask = make_soft_role_feature_mask(pair_ranking, pair_probs, threshold=float(threshold), n_roles=len(ROLE_NAMES), d_sae=int(sae.cfg.d_sae))
                if int((role_feature_mask > 0).sum().item()) == 0:
                    continue
                metrics = compute_faithfulness(sae, dataset, role_feature_mask=role_feature_mask, full_logit_diff=full_logit_diff, batch_size=batch_size)
                rows.append({"baseline": baseline_name, "selection": "soft_threshold", "k": int(metrics["active_nodes"]), "threshold": float(threshold), "gate_sum": float(role_feature_mask.sum().item()), "mean_nonzero_gate": mean_nonzero_gate(role_feature_mask), **metrics})
    results = pd.DataFrame(rows)
    if output_csv_path is not None:
        results.to_csv(output_csv_path, index=False)
        print(f"Saved results to {output_csv_path}")
    return results

def plot_role_pareto(results, output_path=None, title="Role-conditioned SAE circuit Pareto"):
    fig, ax = plt.subplots(figsize=(9, 5.4))
    plot_df = results[~results["baseline"].isin(["all_features", "zero_features"])].copy()
    for (baseline_name, selection), group in plot_df.groupby(["baseline", "selection"], dropna=False):
        ordered = group.sort_values("active_nodes")
        label = baseline_name if selection in (None, "hard_topk") else f"{baseline_name} ({selection})"
        ax.plot(ordered["active_nodes"], ordered["faithfulness"], marker="o", linewidth=1.5, label=label)
    zero_rows = results[results["baseline"] == "zero_features"]
    if len(zero_rows):
        ax.axhline(float(zero_rows.iloc[0]["faithfulness"]), color="tab:red", linestyle="--", linewidth=1.0, label="zero features")
    ax.axhline(1.0, color="gray", linestyle=":", linewidth=1.2, label="full model / all features")
    ax.set_xscale("log")
    ax.set_xlabel("Circuit size (active feature nodes)")
    ax.set_ylabel("Faithfulness (masked logit diff / full logit diff)")
    ax.set_title(title)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=7)
    fig.tight_layout()
    if output_path is not None:
        fig.savefig(output_path, dpi=180, bbox_inches="tight")
        print(f"Saved plot to {output_path}")
    plt.show()
    return fig, ax

def summarize_best(results):
    candidates = results[~results["baseline"].isin(["all_features", "zero_features"])].copy()
    best_by_k = candidates.loc[candidates.groupby("active_nodes")["faithfulness"].idxmax()].sort_values("active_nodes")
    best_by_baseline = candidates.loc[candidates.groupby(["baseline", "selection"])["faithfulness"].idxmax()].sort_values("faithfulness", ascending=False)
    return best_by_k, best_by_baseline

def summarize_win_condition(results, faithfulness_floor=0.95):
    candidates = results[~results["baseline"].isin(["all_features", "zero_features"])].copy()
    wins = candidates[candidates["faithfulness"] >= faithfulness_floor].copy()
    if len(wins) == 0:
        return wins
    return wins.sort_values(["active_nodes", "faithfulness"], ascending=[True, False])

## 9. Batch and Gate Utility Helpers

In [ ]:
def make_batches(dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True):
    groups = group_tokenized_rows(dataset)
    all_batches = []
    for items in groups.values():
        items = list(items)
        if shuffle:
            random.shuffle(items)
        for start in range(0, len(items), batch_size):
            chunk = items[start : start + batch_size]
            indices = torch.tensor([item[0] for item in chunk], dtype=torch.long)
            tokens = torch.stack([item[1] for item in chunk], dim=0)
            role_ids = torch.stack([item[2] for item in chunk], dim=0)
            all_batches.append((indices, tokens, role_ids))
    if shuffle:
        random.shuffle(all_batches)
    return all_batches

def mean_nonzero_gate(mask):
    nonzero = mask[mask > 0]
    if nonzero.numel() == 0:
        return 0.0
    return float(nonzero.mean().item())

## 10. Smoke Test

In [ ]:
smoke_stats = cache_role_feature_stats(smoke_dataset, target_sae, cache_path=SMOKE_STATS_CACHE_PATH, batch_size=BATCH_SIZE, force_recompute=True)
smoke_scores, smoke_rankings = make_rankings(smoke_stats, target_sae)

smoke_answer_direction_stats = cache_answer_direction_feature_stats(
    smoke_dataset,
    target_sae,
    cache_path=SMOKE_ANSWER_DIRECTION_STATS_CACHE_PATH,
    batch_size=ANSWER_ATTRIBUTION_BATCH_SIZE,
    force_recompute=True,
)
smoke_answer_scores, smoke_answer_rankings = make_answer_direction_rankings(smoke_answer_direction_stats, target_sae)
smoke_scores.update(smoke_answer_scores)
smoke_rankings.update(smoke_answer_rankings)

smoke_calibrated_scores, smoke_calibrated_rankings = make_global_calibrated_rankings(
    smoke_stats,
    smoke_answer_direction_stats,
    smoke_rankings,
)
smoke_scores.update(smoke_calibrated_scores)
smoke_rankings.update(smoke_calibrated_rankings)

smoke_results = evaluate_ranked_masks(
    smoke_dataset,
    target_sae,
    smoke_rankings,
    global_k_values=SMOKE_GLOBAL_K_VALUES,
    role_k_values=SMOKE_ROLE_K_VALUES,
    batch_size=BATCH_SIZE,
    output_csv_path=SMOKE_RESULTS_CSV_PATH,
)
write_run_manifest("smoke_completed", extra={"smoke_rows": len(smoke_results)})
smoke_results

In [ ]:
plot_role_pareto(smoke_results, output_path=SMOKE_PLOT_PATH, title="Smoke test: v011 invariant feature circuit benchmark")

## 11. Full v11 Invariant Feature Circuit Run

This section computes global and answer-direction feature rankings separately for each optimization environment, then builds stability-aggregated rankings. The validation step compares pooled-train, single-environment, and invariant masks under the same final held-out and counterfactual-family diagnostics.

In [ ]:
def safe_rank_name(name):
    return str(name).replace("global_", "", 1).replace("answer_gradient_", "answer_")

def env_stats_cache_path(env_name, kind):
    return CACHE_DIR / f"{env_name}_{kind}_layer{TARGET_LAYER}_ioi.pt"

def build_global_ranking_bundle(env_name, dataset, force_recompute=False):
    role_stats = cache_role_feature_stats(
        dataset,
        target_sae,
        cache_path=env_stats_cache_path(env_name, "role_feature_stats"),
        batch_size=BATCH_SIZE,
        force_recompute=force_recompute,
    )
    scores, rankings = make_rankings(role_stats, target_sae)

    answer_stats = cache_answer_direction_feature_stats(
        dataset,
        target_sae,
        cache_path=env_stats_cache_path(env_name, "answer_direction_role_feature_stats"),
        batch_size=ANSWER_ATTRIBUTION_BATCH_SIZE,
        force_recompute=force_recompute,
    )
    answer_scores, answer_rankings = make_answer_direction_rankings(answer_stats, target_sae)
    scores.update(answer_scores)
    rankings.update(answer_rankings)

    calibrated_scores, calibrated_rankings = make_global_calibrated_rankings(role_stats, answer_stats, rankings)
    scores.update(calibrated_scores)
    rankings.update(calibrated_rankings)

    return {
        "env_name": env_name,
        "n_prompts": len(dataset),
        "role_stats": role_stats,
        "answer_stats": answer_stats,
        "scores": scores,
        "rankings": rankings,
    }

def dense_rank_positions(ranking, d_sae=EXPECTED_D_SAE):
    ranking = ranking[: int(d_sae)].to(device)
    positions = torch.empty(int(d_sae), dtype=torch.float32, device=device)
    positions[ranking] = torch.arange(ranking.numel(), dtype=torch.float32, device=device)
    return positions

def aggregate_environment_stability_rankings(env_bundles, source_rankings):
    stability_scores = {}
    stability_rankings = {}
    feature_rows = []
    overlap_rows = []

    env_names = list(env_bundles)
    d_sae = int(target_sae.cfg.d_sae)
    denom = max(1, d_sae - 1)

    for source_name in source_rankings:
        available_envs = [env_name for env_name in env_names if source_name in env_bundles[env_name]["rankings"]]
        if len(available_envs) < 2:
            print(f"Skipping {source_name}: only {len(available_envs)} environments available.")
            continue

        rank_stack = torch.stack([
            dense_rank_positions(env_bundles[env_name]["rankings"][source_name], d_sae=d_sae)
            for env_name in available_envs
        ], dim=0)
        borda_stack = 1.0 - rank_stack / denom
        top_vote = (rank_stack < STABILITY_VOTE_TOP_K).float().mean(dim=0)
        pool_vote = (rank_stack < STABILITY_POOL_K).float().mean(dim=0)
        mean_borda = borda_stack.mean(dim=0)
        worst_borda = borda_stack.min(dim=0).values
        rank_std = rank_stack.std(dim=0, unbiased=False)

        short = safe_rank_name(source_name)
        score_vote = 0.55 * top_vote + 0.25 * pool_vote + 0.15 * mean_borda + 0.05 * worst_borda
        score_worst = 0.50 * worst_borda + 0.30 * mean_borda + 0.20 * top_vote
        score_lowvar = mean_borda - 0.15 * (rank_std / denom) + 0.25 * top_vote

        score_map = {
            f"global_stability_vote_{short}": score_vote,
            f"global_stability_worst_{short}": score_worst,
            f"global_stability_lowvar_{short}": score_lowvar,
        }
        for new_name, score in score_map.items():
            stability_scores[new_name] = score
            stability_rankings[new_name] = score.argsort(descending=True)

        for k in STABILITY_TOP_K_VALUES:
            env_sets = {
                env_name: set(env_bundles[env_name]["rankings"][source_name][:k].detach().cpu().tolist())
                for env_name in available_envs
            }
            pairwise_jaccards = []
            for left_index, left_env in enumerate(available_envs):
                for right_env in available_envs[left_index + 1:]:
                    left, right = env_sets[left_env], env_sets[right_env]
                    pairwise_jaccards.append(len(left & right) / max(1, len(left | right)))
            all_intersection = set.intersection(*env_sets.values())
            all_union = set.union(*env_sets.values())
            overlap_rows.append({
                "source_ranking": source_name,
                "k": int(k),
                "n_envs": len(available_envs),
                "all_env_intersection": len(all_intersection),
                "all_env_union": len(all_union),
                "all_env_jaccard": len(all_intersection) / max(1, len(all_union)),
                "mean_pairwise_jaccard": float(pd.Series(pairwise_jaccards).mean()) if pairwise_jaccards else float("nan"),
                "min_pairwise_jaccard": float(pd.Series(pairwise_jaccards).min()) if pairwise_jaccards else float("nan"),
            })

        top_features = stability_rankings[f"global_stability_vote_{short}"][:STABILITY_SUMMARY_TOP_N]
        for rank, feature_id in enumerate(top_features.detach().cpu().tolist(), start=1):
            env_ranks = {
                f"rank_{env_name}": float(rank_stack[env_index, int(feature_id)].detach().cpu().item())
                for env_index, env_name in enumerate(available_envs)
            }
            feature_rows.append({
                "source_ranking": source_name,
                "stability_ranking": f"global_stability_vote_{short}",
                "stable_rank": rank,
                "feature": int(feature_id),
                "top_vote": float(top_vote[int(feature_id)].detach().cpu().item()),
                "pool_vote": float(pool_vote[int(feature_id)].detach().cpu().item()),
                "mean_borda": float(mean_borda[int(feature_id)].detach().cpu().item()),
                "worst_borda": float(worst_borda[int(feature_id)].detach().cpu().item()),
                "rank_std": float(rank_std[int(feature_id)].detach().cpu().item()),
                "stability_score": float(score_vote[int(feature_id)].detach().cpu().item()),
                **env_ranks,
            })

    return stability_scores, stability_rankings, pd.DataFrame(feature_rows), pd.DataFrame(overlap_rows)

train_bundle = build_global_ranking_bundle("pooled_train", train_dataset, force_recompute=False)
train_stats = train_bundle["role_stats"]
answer_direction_stats = train_bundle["answer_stats"]
feature_scores = dict(train_bundle["scores"])
feature_rankings = dict(train_bundle["rankings"])

environment_bundles = {}
for env_name, env_dataset in optimization_env_datasets.items():
    print(f"Building environment-specific rankings for {env_name} ({len(env_dataset)} prompts)")
    environment_bundles[env_name] = build_global_ranking_bundle(env_name, env_dataset, force_recompute=False)

if INCLUDE_SINGLE_ENVIRONMENT_RANKINGS:
    for env_name, bundle in environment_bundles.items():
        if "global_calibrated_rescue_union" in bundle["rankings"]:
            feature_rankings[f"global_env_{env_name}_calibrated_rescue_union"] = bundle["rankings"]["global_calibrated_rescue_union"]

stability_scores, stability_rankings, stability_feature_diagnostics, stability_overlap_diagnostics = aggregate_environment_stability_rankings(
    environment_bundles,
    STABILITY_SOURCE_RANKINGS,
)
feature_scores.update(stability_scores)
feature_rankings.update(stability_rankings)

stability_trace = pd.concat(
    [
        stability_feature_diagnostics.assign(row_type="feature"),
        stability_overlap_diagnostics.assign(row_type="overlap"),
    ],
    ignore_index=True,
    sort=False,
)
stability_trace.to_csv(INVARIANT_TRACE_CSV_PATH, index=False)
print(f"Saved invariant feature stability diagnostics to {INVARIANT_TRACE_CSV_PATH}")

print("Environment top-K overlap diagnostics:")
display(stability_overlap_diagnostics.sort_values(["source_ranking", "k"]).head(40))

print("Top pooled calibrated features:")
display(describe_global_features(
    feature_rankings["global_calibrated_rescue_union"],
    score_table={
        "activation": train_stats["mean_abs_all"],
        "answer_abs": answer_direction_stats["mean_abs_all"],
        "answer_support": answer_direction_stats["support_all"],
        "calibrated_rescue": feature_scores["global_calibrated_rescue_union"],
    },
    limit=16,
))

stable_name = "global_stability_vote_calibrated_rescue_union"
if stable_name in feature_rankings:
    print("Top invariant calibrated features:")
    display(stability_feature_diagnostics[stability_feature_diagnostics["stability_ranking"] == stable_name].head(20))

for role_name, count in zip(ROLE_NAMES, train_stats["count_by_role"].detach().cpu().tolist()):
    print(f"{role_name:>18}: {int(count)} pooled-train token positions")
print(f"Mean pooled-train full-model logit diff used for attribution: {answer_direction_stats['mean_full_logit_diff']:.4f}")
print(f"Mean answer direction norm: {answer_direction_stats['mean_answer_direction_norm']:.4f}")

In [ ]:
def make_global_env_training_state(env_datasets, batch_size=TRAIN_BATCH_SIZE):
    env_states = []
    for env_name, dataset in env_datasets.items():
        train_df = dataset.reset_index(drop=True).copy()
        full_logits = run_logits(train_df, batch_size=BATCH_SIZE, detach=True)
        full_diffs = example_logit_diffs(full_logits, train_df).to(device)
        clean_ids = torch.tensor(train_df["answer_clean_id"].to_list(), dtype=torch.long, device=device)
        corrupt_ids = torch.tensor(train_df["answer_corrupt_id"].to_list(), dtype=torch.long, device=device)
        env_states.append({
            "name": env_name,
            "dataset": train_df,
            "full_diffs": full_diffs,
            "clean_ids": clean_ids,
            "corrupt_ids": corrupt_ids,
            "batches": make_batches(train_df, batch_size=batch_size, shuffle=True),
        })
    return env_states

def collect_token_rows(dataset):
    tokens_by_row = [None] * len(dataset)
    for items in group_tokenized_rows(dataset).values():
        for index, tokens, _role_ids in items:
            tokens_by_row[index] = tokens
    if any(x is None for x in tokens_by_row):
        raise ValueError("Missing tokenized rows in global family state.")
    return tokens_by_row

def make_family_batches(family_ids, batch_families=COUNTERFACTUAL_BATCH_FAMILIES, shuffle=True):
    family_ids = list(family_ids)
    if shuffle:
        random.shuffle(family_ids)
    return [family_ids[start : start + batch_families] for start in range(0, len(family_ids), batch_families)]

def make_global_counterfactual_family_state(dataset, batch_families=COUNTERFACTUAL_BATCH_FAMILIES):
    family_df = dataset.reset_index(drop=True).copy()
    if "family_id" not in family_df or family_df["family_id"].isna().any():
        raise ValueError("Counterfactual family dataset must have non-null family_id values.")
    full_logits = run_logits(family_df, batch_size=BATCH_SIZE, detach=True)
    full_diffs = example_logit_diffs(full_logits, family_df).to(device)
    clean_ids = torch.tensor(family_df["answer_clean_id"].to_list(), dtype=torch.long, device=device)
    corrupt_ids = torch.tensor(family_df["answer_corrupt_id"].to_list(), dtype=torch.long, device=device)
    tokens_by_row = collect_token_rows(family_df)
    family_to_indices = {
        family_id: group.index.to_list()
        for family_id, group in family_df.groupby("family_id", sort=False)
    }
    family_ids = list(family_to_indices)
    return {
        "name": str(family_df["split"].iloc[0]),
        "dataset": family_df,
        "full_diffs": full_diffs,
        "clean_ids": clean_ids,
        "corrupt_ids": corrupt_ids,
        "tokens_by_row": tokens_by_row,
        "family_to_indices": family_to_indices,
        "family_ids": family_ids,
        "family_batches": make_family_batches(family_ids, batch_families=batch_families, shuffle=True),
    }

def logits_for_global_token_batch(tokens, sae, feature_mask):
    hook_name = hook_name_for_layer(TARGET_LAYER)
    fwd_hooks = [(hook_name, partial(sae_role_mask_hook, sae=sae, feature_mask=feature_mask, preserve_error=True))]
    with model.hooks(fwd_hooks=fwd_hooks):
        return model(tokens)[:, -1, :]

def logits_for_global_state_indices(state, indices, sae, feature_mask):
    indices = [int(x) for x in indices]
    grouped = defaultdict(list)
    for index in indices:
        grouped[int(state["tokens_by_row"][index].numel())].append(index)
    logits_by_index = {}
    for grouped_indices in grouped.values():
        tokens = torch.stack([state["tokens_by_row"][idx] for idx in grouped_indices], dim=0)
        logits = logits_for_global_token_batch(tokens, sae, feature_mask)
        for idx, row_logits in zip(grouped_indices, logits):
            logits_by_index[idx] = row_logits
    return torch.stack([logits_by_index[idx] for idx in indices], dim=0)

def family_batch_indices_and_labels(state, family_ids):
    indices = []
    labels = []
    for family_id in family_ids:
        family_indices = state["family_to_indices"][family_id]
        indices.extend(family_indices)
        labels.extend([family_id] * len(family_indices))
    return indices, labels

def global_counterfactual_family_losses(state, family_ids, sae, feature_mask):
    row_indices, family_labels = family_batch_indices_and_labels(state, family_ids)
    logits = logits_for_global_state_indices(state, row_indices, sae, feature_mask)
    index_tensor = torch.tensor(row_indices, dtype=torch.long, device=device)
    batch_rows = torch.arange(logits.shape[0], device=device)
    masked_diffs = logits[batch_rows, state["clean_ids"][index_tensor]] - logits[batch_rows, state["corrupt_ids"][index_tensor]]
    full_diffs = state["full_diffs"][index_tensor].detach()
    ratios = masked_diffs / (full_diffs + 1e-6)

    ratio_loss = (ratios - 1.0).pow(2).mean()
    family_mean_losses = []
    family_variance_losses = []
    family_worst_errors = []
    for family_id in family_ids:
        positions = [idx for idx, label in enumerate(family_labels) if label == family_id]
        position_tensor = torch.tensor(positions, dtype=torch.long, device=device)
        family_ratios = ratios[position_tensor]
        family_mean_losses.append((family_ratios.mean() - 1.0).pow(2))
        family_variance_losses.append(family_ratios.var(unbiased=False))
        family_worst_errors.append((family_ratios - 1.0).abs().max())
    return {
        "ratio_loss": ratio_loss,
        "family_mean_loss": torch.stack(family_mean_losses).mean(),
        "name_variance_loss": torch.stack(family_variance_losses).mean(),
        "worst_family_loss": torch.stack(family_worst_errors).max().pow(2),
        "mean_ratio": ratios.mean(),
        "min_ratio": ratios.min(),
        "max_ratio": ratios.max(),
    }

def train_global_soft_mask(env_datasets, sae, candidate_features, candidate_scores=None, steps=GLOBAL_SOFT_STEPS, lr=GLOBAL_SOFT_LR, lambda_size=GLOBAL_SOFT_LAMBDA, batch_size=TRAIN_BATCH_SIZE):
    env_states = make_global_env_training_state(env_datasets, batch_size=batch_size)
    family_state = make_global_counterfactual_family_state(name_cf_train_dataset, batch_families=COUNTERFACTUAL_BATCH_FAMILIES)
    candidate_features = candidate_features[:GLOBAL_SOFT_CANDIDATE_FEATURES].to(device)
    if candidate_scores is None:
        candidate_scores = torch.ones(candidate_features.numel(), device=device)
    candidate_scores = candidate_scores[:candidate_features.numel()].to(device=device, dtype=torch.float32)
    candidate_prior = candidate_scores / candidate_scores.max().clamp_min(ANSWER_ATTRIBUTION_SCORE_EPS)
    prior_std = candidate_prior.std(unbiased=False).clamp_min(ANSWER_ATTRIBUTION_SCORE_EPS)
    initial_logits = (1.25 + (candidate_prior - candidate_prior.mean()) / prior_std).clamp(-2.5, 3.0)
    gain_logits = torch.nn.Parameter(initial_logits.clone())
    optimizer = torch.optim.Adam([gain_logits], lr=lr)
    trace = []

    for step in tqdm(range(steps), desc=f"Global soft gains lambda={lambda_size}"):
        for state in env_states:
            if step % max(1, len(state["batches"])) == 0:
                state["batches"] = make_batches(state["dataset"], batch_size=batch_size, shuffle=True)
        if step % max(1, len(family_state["family_batches"])) == 0:
            family_state["family_batches"] = make_family_batches(
                family_state["family_ids"],
                batch_families=COUNTERFACTUAL_BATCH_FAMILIES,
                shuffle=True,
            )

        optimizer.zero_grad(set_to_none=True)
        with torch.enable_grad():
            feature_gains = GLOBAL_SOFT_MAX_GAIN * torch.sigmoid(gain_logits)
            feature_mask = dense_global_mask_from_candidate_values(candidate_features, feature_gains)
            env_faithfulnesses = []
            env_mean_losses = []
            env_ratio_losses = []
            env_loss_by_name = {}

            for state in env_states:
                indices, tokens, _role_ids = state["batches"][step % len(state["batches"])]
                indices = indices.to(device)
                logits = logits_for_global_token_batch(tokens, sae, feature_mask)
                row_indices = torch.arange(tokens.shape[0], device=device)
                masked_diffs = logits[row_indices, state["clean_ids"][indices]] - logits[row_indices, state["corrupt_ids"][indices]]
                full_batch_diffs = state["full_diffs"][indices].detach()
                faithfulness = masked_diffs.mean() / (full_batch_diffs.mean().detach() + 1e-6)
                example_ratios = masked_diffs / (full_batch_diffs + 1e-6)
                env_faithfulnesses.append(faithfulness)
                env_mean_losses.append((faithfulness - 1.0).pow(2))
                env_ratio_losses.append((example_ratios - 1.0).pow(2).mean())
                env_loss_by_name[state["name"]] = faithfulness

            family_ids = family_state["family_batches"][step % len(family_state["family_batches"])]
            cf_losses = global_counterfactual_family_losses(family_state, family_ids, sae, feature_mask)

            env_faithfulnesses = torch.stack(env_faithfulnesses)
            reconstruction_loss = torch.stack(env_mean_losses).mean()
            example_ratio_loss = torch.stack(env_ratio_losses).mean()
            worst_env_loss = (env_faithfulnesses - 1.0).abs().max().pow(2)
            variance_loss = env_faithfulnesses.var(unbiased=False)
            size_loss = (feature_gains / GLOBAL_SOFT_MAX_GAIN).mean()
            attribution_prior_loss = -((feature_gains * candidate_prior).sum() / feature_gains.sum().clamp_min(ANSWER_ATTRIBUTION_SCORE_EPS))
            gain_anchor_loss = (feature_gains - 1.0).pow(2).mean()
            loss = (
                reconstruction_loss
                + ROBUST_EXAMPLE_RATIO_WEIGHT * example_ratio_loss
                + ROBUST_WORST_ENV_WEIGHT * worst_env_loss
                + ROBUST_VARIANCE_WEIGHT * variance_loss
                + COUNTERFACTUAL_RATIO_WEIGHT * cf_losses["ratio_loss"]
                + COUNTERFACTUAL_FAMILY_MEAN_WEIGHT * cf_losses["family_mean_loss"]
                + COUNTERFACTUAL_NAME_VARIANCE_WEIGHT * cf_losses["name_variance_loss"]
                + COUNTERFACTUAL_WORST_FAMILY_WEIGHT * cf_losses["worst_family_loss"]
                + GLOBAL_ATTRIBUTION_PRIOR_WEIGHT * attribution_prior_loss
                + GLOBAL_GAIN_ANCHOR_WEIGHT * gain_anchor_loss
                + lambda_size * size_loss
            )

        loss.backward()
        optimizer.step()

        if step % 10 == 0 or step == steps - 1:
            row = {
                "lambda": float(lambda_size),
                "step": int(step),
                "loss": float(loss.detach().item()),
                "reconstruction_loss": float(reconstruction_loss.detach().item()),
                "example_ratio_loss": float(example_ratio_loss.detach().item()),
                "worst_env_loss": float(worst_env_loss.detach().item()),
                "variance_loss": float(variance_loss.detach().item()),
                "counterfactual_ratio_loss": float(cf_losses["ratio_loss"].detach().item()),
                "counterfactual_family_mean_loss": float(cf_losses["family_mean_loss"].detach().item()),
                "counterfactual_name_variance_loss": float(cf_losses["name_variance_loss"].detach().item()),
                "counterfactual_worst_family_loss": float(cf_losses["worst_family_loss"].detach().item()),
                "counterfactual_mean_ratio": float(cf_losses["mean_ratio"].detach().item()),
                "counterfactual_min_ratio": float(cf_losses["min_ratio"].detach().item()),
                "counterfactual_max_ratio": float(cf_losses["max_ratio"].detach().item()),
                "size_loss": float(size_loss.detach().item()),
                "gain_anchor_loss": float(gain_anchor_loss.detach().item()),
                "attribution_prior_loss": float(attribution_prior_loss.detach().item()),
                "answer_attribution_prior_mass": float(((feature_gains * candidate_prior).sum() / feature_gains.sum().clamp_min(ANSWER_ATTRIBUTION_SCORE_EPS)).detach().item()),
                "mean_batch_faithfulness": float(env_faithfulnesses.detach().mean().item()),
                "min_batch_faithfulness": float(env_faithfulnesses.detach().min().item()),
                "max_batch_faithfulness": float(env_faithfulnesses.detach().max().item()),
                "mean_feature_gain": float(feature_gains.detach().mean().item()),
                "max_feature_gain": float(feature_gains.detach().max().item()),
                "sum_feature_gain": float(feature_gains.detach().sum().item()),
            }
            for env_name, faithfulness in env_loss_by_name.items():
                row[f"faithfulness_{env_name}"] = float(faithfulness.detach().item())
            trace.append(row)

    learned_gains = (GLOBAL_SOFT_MAX_GAIN * torch.sigmoid(gain_logits)).detach()
    learned_order = torch.argsort(learned_gains, descending=True)
    return candidate_features[learned_order].detach(), learned_gains[learned_order].detach(), pd.DataFrame(trace)

In [ ]:
global_soft_trace = pd.DataFrame()
global_soft_gate_specs = {}
print("V11 skips learned global soft gains. The experiment compares pooled, single-environment, and invariant hard feature rankings.")

mask_payload = {
    "feature_rankings": {name: ranking.detach().cpu() for name, ranking in feature_rankings.items()},
    "stability_scores": {name: score.detach().cpu() for name, score in stability_scores.items()},
    "environment_feature_rankings": {
        env_name: {
            ranking_name: ranking.detach().cpu()
            for ranking_name, ranking in bundle["rankings"].items()
            if ranking_name.startswith("global_")
        }
        for env_name, bundle in environment_bundles.items()
    },
    "stability_feature_diagnostics_path": str(INVARIANT_TRACE_CSV_PATH),
    "role_names": ROLE_NAMES,
    "optimization_env_names": OPTIMIZATION_ENV_NAMES,
    "final_validation_splits": FINAL_VALIDATION_SPLITS,
    "counterfactual_family_splits": list(counterfactual_family_datasets),
    "answer_direction_stats_cache_path": str(ANSWER_DIRECTION_STATS_CACHE_PATH),
    "run_version": RUN_VERSION,
}
torch.save(mask_payload, MASKS_PATH)
print(f"Saved compact invariant feature circuit payload to {MASKS_PATH}")

In [ ]:
def make_mask_id(baseline, selection, k=None, threshold=None, repeat=None):
    pieces = [str(baseline), str(selection)]
    if k is not None:
        pieces.append(f"k={int(k)}")
    if threshold is not None:
        pieces.append(f"thr={float(threshold):.2f}")
    if repeat is not None:
        pieces.append(f"repeat={int(repeat)}")
    return "|".join(pieces)

def build_validation_mask_specs(sae, rankings, global_soft_specs):
    d_sae = int(sae.cfg.d_sae)
    specs = []

    def add_spec(baseline, selection, k=None, threshold=None, feature_mask=None, role_feature_mask=None, repeat=None):
        if role_feature_mask is not None:
            active_nodes = int((role_feature_mask > 0).sum().item())
            unique_features = int(((role_feature_mask > 0).sum(dim=0) > 0).sum().item())
            gate_sum = float(role_feature_mask.sum().item())
            mean_gate = mean_nonzero_gate(role_feature_mask)
            mask_type = "role_feature"
        elif feature_mask is not None:
            active_nodes = int((feature_mask > 0).sum().item())
            unique_features = active_nodes
            gate_sum = float(feature_mask.sum().item())
            mean_gate = mean_nonzero_gate(feature_mask)
            mask_type = "global_feature"
        else:
            active_nodes = 0
            unique_features = 0
            gate_sum = 0.0
            mean_gate = 0.0
            mask_type = "none"
        specs.append({
            "mask_id": make_mask_id(baseline, selection, k=k, threshold=threshold, repeat=repeat),
            "baseline": baseline,
            "selection": selection,
            "k": None if k is None else int(k),
            "threshold": None if threshold is None else float(threshold),
            "repeat": repeat,
            "feature_mask": feature_mask,
            "role_feature_mask": role_feature_mask,
            "active_nodes_spec": active_nodes,
            "unique_features_spec": unique_features,
            "gate_sum": gate_sum,
            "mean_nonzero_gate": mean_gate,
            "mask_type_spec": mask_type,
        })

    add_spec("all_features", "reference", k=d_sae, feature_mask=torch.ones(d_sae, device=device))
    add_spec("zero_features", "reference", k=0, feature_mask=torch.zeros(d_sae, device=device))

    validation_global_names = list(VALIDATION_GLOBAL_BASELINES)
    if INCLUDE_SINGLE_ENVIRONMENT_RANKINGS:
        validation_global_names.extend(sorted([name for name in rankings if name.startswith("global_env_")]))
    global_baselines = [name for name in validation_global_names if name in rankings]
    missing_global_baselines = [name for name in validation_global_names if name not in rankings]
    if missing_global_baselines:
        print("Skipping missing global baselines:", missing_global_baselines)
    for baseline_name in global_baselines:
        for k in GLOBAL_K_VALUES:
            add_spec(
                baseline_name,
                "hard_topk",
                k=k,
                feature_mask=make_global_feature_mask(rankings[baseline_name], k, d_sae=d_sae),
            )

    static_role_baselines = ["role_mean_abs_all_roles", "role_wanda_all_roles"]
    for baseline_name in static_role_baselines:
        if baseline_name not in rankings:
            continue
        for k in [1_000, 2_000]:
            add_spec(
                baseline_name,
                "hard_topk",
                k=k,
                role_feature_mask=make_role_feature_mask(rankings[baseline_name], k, n_roles=len(ROLE_NAMES), d_sae=d_sae),
            )

    for baseline_name, spec in sorted(global_soft_specs.items()):
        feature_ranking = spec["feature_ranking"]
        feature_gains = spec["feature_gains"]
        for k in GLOBAL_SOFT_TOP_K_VALUES:
            add_spec(
                baseline_name,
                "soft_topk",
                k=k,
                feature_mask=make_soft_global_feature_mask(feature_ranking, feature_gains, top_k=int(k), d_sae=d_sae),
            )
        for threshold in GLOBAL_SOFT_THRESHOLD_VALUES:
            feature_mask = make_soft_global_feature_mask(feature_ranking, feature_gains, threshold=float(threshold), d_sae=d_sae)
            if int((feature_mask > 0).sum().item()) == 0:
                continue
            add_spec(
                baseline_name,
                "soft_threshold",
                k=int((feature_mask > 0).sum().item()),
                threshold=float(threshold),
                feature_mask=feature_mask,
            )

    candidate_pool = rankings["global_calibrated_rescue_union"][:GLOBAL_SOFT_CANDIDATE_FEATURES].detach().cpu()
    cpu_generator = torch.Generator(device="cpu")
    for repeat in range(RANDOM_CONTROL_REPEATS):
        cpu_generator.manual_seed(SEED + 30_000 + repeat)
        shuffled_pool = candidate_pool[torch.randperm(candidate_pool.numel(), generator=cpu_generator)].to(device)
        for k in RANDOM_CONTROL_K_VALUES:
            add_spec(
                "random_global_features_from_calibrated_pool",
                "random_topk",
                k=k,
                feature_mask=make_global_feature_mask(shuffled_pool, k, d_sae=d_sae),
                repeat=repeat,
            )

    return specs

def evaluate_mask_specs_on_splits(split_datasets, sae, mask_specs, batch_size=BATCH_SIZE):
    rows = []
    for split_name, dataset in split_datasets.items():
        full_logits = run_logits(dataset, batch_size=batch_size, detach=True)
        full_logit_diff = mean_logit_diff(full_logits, dataset)
        split_kind = "final_validation" if split_name in FINAL_VALIDATION_SPLITS else "optimization_env"
        print(f"{split_name}: full-model logit diff = {float(full_logit_diff.item()):.4f}")
        for spec in tqdm(mask_specs, desc=f"Evaluating {split_name}"):
            metrics = compute_faithfulness(
                sae,
                dataset,
                feature_mask=spec["feature_mask"],
                role_feature_mask=spec["role_feature_mask"],
                full_logit_diff=full_logit_diff,
                batch_size=batch_size,
            )
            faithfulness = metrics["faithfulness"]
            row = {
                "split": split_name,
                "split_kind": split_kind,
                "n_prompts": len(dataset),
                "mask_id": spec["mask_id"],
                "baseline": spec["baseline"],
                "selection": spec["selection"],
                "k": spec["k"],
                "threshold": spec["threshold"],
                "repeat": spec["repeat"],
                "gate_sum": spec["gate_sum"],
                "mean_nonzero_gate": spec["mean_nonzero_gate"],
                **metrics,
            }
            row["faithfulness_abs_error"] = abs(faithfulness - 1.0)
            row["within_5pct_band"] = FAITHFULNESS_BAND_LOW <= faithfulness <= FAITHFULNESS_BAND_HIGH
            row["over_recovers"] = faithfulness > FAITHFULNESS_BAND_HIGH
            row["under_recovers"] = faithfulness < FAITHFULNESS_BAND_LOW
            rows.append(row)
    return pd.DataFrame(rows)

def summarize_split_table(results, top_summary):
    top_ids = top_summary.head(12)["mask_id"].tolist()
    pivot = results[results["mask_id"].isin(top_ids)].pivot_table(
        index=["baseline", "selection", "k", "threshold"],
        columns="split",
        values="faithfulness",
        aggfunc="mean",
    )
    return pivot.reset_index()

def mask_display_label(row):
    baseline = str(row["baseline"])
    selection = str(row["selection"])
    k = row.get("k")
    threshold = row.get("threshold")
    if pd.notna(threshold):
        suffix = f"thr={float(threshold):.2f}"
    elif pd.notna(k):
        suffix = f"K={int(k)}"
    else:
        suffix = selection
    return f"{baseline}\n{suffix}"

def plot_validation_pareto(summary, output_path=None):
    fig, ax = plt.subplots(figsize=(10, 6))
    plot_df = summary[~summary["baseline"].isin(["all_features", "zero_features"])].copy()
    families = [
        ("global_activation_mean_abs", "tab:blue"),
        ("global_answer_gradient_abs", "tab:orange"),
        ("global_calibrated_rescue_union", "tab:green"),
        ("global_calibrated_borda_activation_answer", "tab:purple"),
        ("global_stability_vote_calibrated_rescue_union", "tab:red"),
        ("global_stability_worst_calibrated_rescue_union", "tab:brown"),
        ("global_stability_lowvar_calibrated_rescue_union", "tab:pink"),
    ]
    for baseline_name, color in families:
        group = plot_df[plot_df["baseline"] == baseline_name].sort_values("active_nodes")
        if len(group):
            ax.plot(group["active_nodes"], group["final_mean_faithfulness"], marker="o", linewidth=1.5, color=color, label=baseline_name)
    env_rows = plot_df[plot_df["baseline"].str.startswith("global_env_", na=False)].copy()
    if len(env_rows):
        for baseline_name, group in env_rows.groupby("baseline"):
            ordered = group.sort_values("active_nodes")
            ax.scatter(ordered["active_nodes"], ordered["final_mean_faithfulness"], s=18, alpha=0.35, label=baseline_name)
    ax.axhspan(FAITHFULNESS_BAND_LOW, FAITHFULNESS_BAND_HIGH, color="gray", alpha=0.12, label="5pct band")
    ax.axhline(1.0, color="gray", linestyle=":", linewidth=1.2)
    ax.set_xscale("log")
    ax.set_xlabel("Circuit size (active SAE features)")
    ax.set_ylabel("Final mean faithfulness")
    ax.set_title("v11 invariant feature circuits: final-validation Pareto")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=7)
    fig.tight_layout()
    if output_path is not None:
        fig.savefig(output_path, dpi=180, bbox_inches="tight")
        print(f"Saved validation Pareto plot to {output_path}")
    plt.show()
    return fig, ax

def plot_generalization_heatmap(results, summary, output_path=None, top_n=16):
    top_ids = summary.head(top_n)["mask_id"].tolist()
    label_map = {row["mask_id"]: mask_display_label(row) for _, row in summary[summary["mask_id"].isin(top_ids)].iterrows()}
    plot_df = results[results["mask_id"].isin(top_ids)].copy()
    plot_df["mask_label"] = plot_df["mask_id"].map(label_map)
    split_order = OPTIMIZATION_ENV_NAMES + FINAL_VALIDATION_SPLITS
    row_order = [label_map[mask_id] for mask_id in top_ids if mask_id in label_map]
    pivot = plot_df.pivot_table(index="mask_label", columns="split", values="faithfulness", aggfunc="mean").reindex(index=row_order, columns=split_order)
    values = pivot.to_numpy(dtype=float)
    fig, ax = plt.subplots(figsize=(12, max(5, 0.45 * len(row_order))))
    im = ax.imshow(values, aspect="auto", cmap="coolwarm", vmin=0.55, vmax=1.15)
    ax.set_xticks(range(len(split_order)))
    ax.set_xticklabels(split_order, rotation=35, ha="right")
    ax.set_yticks(range(len(row_order)))
    ax.set_yticklabels(row_order, fontsize=7)
    ax.set_title("V11 faithfulness by optimization and final split")
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            if pd.notna(values[i, j]):
                ax.text(j, i, f"{values[i, j]:.2f}", ha="center", va="center", fontsize=6, color="black")
    fig.colorbar(im, ax=ax, label="Faithfulness")
    fig.tight_layout()
    if output_path is not None:
        fig.savefig(output_path, dpi=180, bbox_inches="tight")
        print(f"Saved heatmap to {output_path}")
    plt.show()
    return fig, ax

In [ ]:
def summarize_validation_results(results):
    group_cols = [
        "mask_id",
        "baseline",
        "selection",
        "k",
        "threshold",
        "repeat",
        "active_nodes",
        "unique_features",
        "mask_type",
        "gate_sum",
        "mean_nonzero_gate",
    ]
    candidates = results[~results["baseline"].isin(["all_features", "zero_features"])].copy()
    summary = candidates.groupby(group_cols, dropna=False).agg(
        mean_faithfulness=("faithfulness", "mean"),
        min_faithfulness=("faithfulness", "min"),
        max_faithfulness=("faithfulness", "max"),
        mean_abs_error=("faithfulness_abs_error", "mean"),
        max_abs_error=("faithfulness_abs_error", "max"),
        in_band_rate=("within_5pct_band", "mean"),
        n_splits=("split", "nunique"),
        mean_masked_logit_diff=("masked_logit_diff", "mean"),
        mean_full_logit_diff=("full_logit_diff", "mean"),
    ).reset_index()

    optimization = candidates[candidates["split"].isin(OPTIMIZATION_ENV_NAMES)].copy()
    optimization_summary = optimization.groupby(group_cols, dropna=False).agg(
        opt_mean_faithfulness=("faithfulness", "mean"),
        opt_min_faithfulness=("faithfulness", "min"),
        opt_max_faithfulness=("faithfulness", "max"),
        opt_mean_abs_error=("faithfulness_abs_error", "mean"),
        opt_max_abs_error=("faithfulness_abs_error", "max"),
        opt_in_band_rate=("within_5pct_band", "mean"),
        opt_n_splits=("split", "nunique"),
    ).reset_index()
    summary = summary.merge(optimization_summary, on=group_cols, how="left")

    final = candidates[candidates["split"].isin(FINAL_VALIDATION_SPLITS)].copy()
    final_summary = final.groupby(group_cols, dropna=False).agg(
        final_mean_faithfulness=("faithfulness", "mean"),
        final_min_faithfulness=("faithfulness", "min"),
        final_max_faithfulness=("faithfulness", "max"),
        final_mean_abs_error=("faithfulness_abs_error", "mean"),
        final_max_abs_error=("faithfulness_abs_error", "max"),
        final_in_band_rate=("within_5pct_band", "mean"),
        final_n_splits=("split", "nunique"),
    ).reset_index()
    summary = summary.merge(final_summary, on=group_cols, how="left")
    summary["passes_optimization_splits"] = summary["opt_in_band_rate"] == 1.0
    summary["passes_all_splits"] = summary["in_band_rate"] == 1.0
    summary["passes_final_splits"] = summary["final_in_band_rate"] == 1.0
    summary["compression_vs_global_500"] = 500.0 / summary["active_nodes"].clip(lower=1)
    summary["final_faithfulness_range"] = summary["final_max_faithfulness"] - summary["final_min_faithfulness"]
    summary["selection_score"] = summary["opt_max_abs_error"] + 0.00002 * summary["active_nodes"].clip(lower=1)
    summary = summary.sort_values(
        ["passes_optimization_splits", "opt_max_abs_error", "selection_score", "active_nodes", "final_max_abs_error"],
        ascending=[False, True, True, True, True],
    ).reset_index(drop=True)
    return summary

def evaluate_counterfactual_family_diagnostics(family_datasets, sae, mask_specs, batch_size=BATCH_SIZE):
    rows = []
    for split_name, dataset in family_datasets.items():
        dataset = dataset.reset_index(drop=True).copy()
        full_logits = run_logits(dataset, batch_size=batch_size, detach=True)
        full_diffs = example_logit_diffs(full_logits, dataset)
        print(f"{split_name}: evaluating {dataset['family_id'].nunique()} counterfactual families")
        for spec in tqdm(mask_specs, desc=f"Counterfactual families {split_name}"):
            masked_logits = run_logits(
                dataset,
                batch_size=batch_size,
                sae=sae,
                feature_mask=spec["feature_mask"],
                role_feature_mask=spec["role_feature_mask"],
                use_sae_substitution=True,
                preserve_error=True,
                detach=True,
            )
            masked_diffs = example_logit_diffs(masked_logits, dataset)
            ratios = (masked_diffs / (full_diffs + 1e-6)).detach().cpu()
            diag_df = dataset[["family_id", "family_member", "name_pair", "template", "split"]].copy()
            diag_df["ratio"] = ratios.numpy()
            for family_id, group in diag_df.groupby("family_id", sort=False):
                family_ratios = group["ratio"].to_numpy(dtype=float)
                family_mean = float(family_ratios.mean())
                family_std = float(family_ratios.std(ddof=0))
                family_min = float(family_ratios.min())
                family_max = float(family_ratios.max())
                rows.append({
                    "split": split_name,
                    "split_kind": "optimization_counterfactual" if split_name == "name_cf_train" else "final_counterfactual",
                    "family_id": family_id,
                    "n_family_members": len(group),
                    "template": group["template"].iloc[0],
                    "mask_id": spec["mask_id"],
                    "baseline": spec["baseline"],
                    "selection": spec["selection"],
                    "k": spec["k"],
                    "threshold": spec["threshold"],
                    "repeat": spec["repeat"],
                    "active_nodes": spec["active_nodes_spec"],
                    "unique_features": spec["unique_features_spec"],
                    "family_mean_ratio": family_mean,
                    "family_ratio_std": family_std,
                    "family_min_ratio": family_min,
                    "family_max_ratio": family_max,
                    "family_mean_abs_error": abs(family_mean - 1.0),
                    "family_max_abs_error": max(abs(family_min - 1.0), abs(family_max - 1.0)),
                    "within_5pct_family_mean": FAITHFULNESS_BAND_LOW <= family_mean <= FAITHFULNESS_BAND_HIGH,
                    "within_5pct_all_members": bool(((family_ratios >= FAITHFULNESS_BAND_LOW) & (family_ratios <= FAITHFULNESS_BAND_HIGH)).all()),
                })
    return pd.DataFrame(rows)

def add_counterfactual_summary_metrics(summary, family_results):
    if len(family_results) == 0:
        return summary
    train = family_results[family_results["split"] == "name_cf_train"].copy()
    train_summary = train.groupby("mask_id").agg(
        cf_train_mean_family_ratio=("family_mean_ratio", "mean"),
        cf_train_mean_family_abs_error=("family_mean_abs_error", "mean"),
        cf_train_worst_family_abs_error=("family_max_abs_error", "max"),
        cf_train_mean_name_std=("family_ratio_std", "mean"),
        cf_train_worst_name_std=("family_ratio_std", "max"),
        cf_train_family_mean_in_band_rate=("within_5pct_family_mean", "mean"),
        cf_train_all_members_in_band_rate=("within_5pct_all_members", "mean"),
    ).reset_index()

    final = family_results[family_results["split"] != "name_cf_train"].copy()
    final_summary = final.groupby("mask_id").agg(
        cf_final_mean_family_ratio=("family_mean_ratio", "mean"),
        cf_final_mean_family_abs_error=("family_mean_abs_error", "mean"),
        cf_final_worst_family_abs_error=("family_max_abs_error", "max"),
        cf_final_mean_name_std=("family_ratio_std", "mean"),
        cf_final_worst_name_std=("family_ratio_std", "max"),
        cf_final_family_mean_in_band_rate=("within_5pct_family_mean", "mean"),
        cf_final_all_members_in_band_rate=("within_5pct_all_members", "mean"),
    ).reset_index()

    merged = summary.merge(train_summary, on="mask_id", how="left").merge(final_summary, on="mask_id", how="left")
    for column in [
        "cf_train_mean_family_abs_error",
        "cf_train_worst_family_abs_error",
        "cf_train_mean_name_std",
        "cf_train_worst_name_std",
        "cf_train_family_mean_in_band_rate",
        "cf_train_all_members_in_band_rate",
        "cf_final_mean_family_abs_error",
        "cf_final_worst_family_abs_error",
        "cf_final_mean_name_std",
        "cf_final_worst_name_std",
        "cf_final_family_mean_in_band_rate",
        "cf_final_all_members_in_band_rate",
    ]:
        if column in merged:
            merged[column] = merged[column].fillna(float("inf") if "rate" not in column else 0.0)
    merged["invariant_selection_score"] = (
        merged["opt_max_abs_error"].fillna(1.0)
        + merged["cf_train_worst_family_abs_error"].fillna(1.0)
        + merged["cf_train_worst_name_std"].fillna(1.0)
        + 0.00002 * merged["active_nodes"].clip(lower=1)
    )
    merged["invariant_diagnostic_score"] = (
        merged["final_max_abs_error"].fillna(1.0)
        + merged["cf_final_worst_family_abs_error"].fillna(1.0)
        + merged["cf_final_worst_name_std"].fillna(1.0)
        + 0.00002 * merged["active_nodes"].clip(lower=1)
    )
    merged = merged.sort_values(
        [
            "passes_optimization_splits",
            "cf_train_all_members_in_band_rate",
            "cf_train_worst_family_abs_error",
            "cf_train_worst_name_std",
            "opt_max_abs_error",
            "active_nodes",
            "final_max_abs_error",
        ],
        ascending=[False, False, True, True, True, True, True],
    ).reset_index(drop=True)
    return merged

def plot_counterfactual_family_heatmap(family_results, summary, output_path=None, top_n=GROUP_DIAGNOSTIC_TOP_N):
    if len(family_results) == 0:
        print("No counterfactual family rows to plot.")
        return None, None
    top_ids = summary.head(top_n)["mask_id"].tolist()
    label_map = {row["mask_id"]: mask_display_label(row) for _, row in summary[summary["mask_id"].isin(top_ids)].iterrows()}
    plot_df = family_results[family_results["mask_id"].isin(top_ids)].copy()
    plot_df["mask_label"] = plot_df["mask_id"].map(label_map)
    aggregate = plot_df.groupby(["mask_label", "split"], sort=False).agg(
        mean_ratio=("family_mean_ratio", "mean"),
        mean_std=("family_ratio_std", "mean"),
    ).reset_index()
    split_order = list(counterfactual_family_datasets.keys())
    row_order = [label_map[mask_id] for mask_id in top_ids if mask_id in label_map]
    pivot_mean = aggregate.pivot(index="mask_label", columns="split", values="mean_ratio").reindex(index=row_order, columns=split_order)
    pivot_std = aggregate.pivot(index="mask_label", columns="split", values="mean_std").reindex(index=row_order, columns=split_order)
    values = pivot_mean.to_numpy(dtype=float)
    std_values = pivot_std.to_numpy(dtype=float)

    fig, ax = plt.subplots(figsize=(10, max(4, 0.55 * len(row_order))))
    im = ax.imshow(values, aspect="auto", cmap="coolwarm", vmin=0.65, vmax=1.25)
    ax.set_xticks(range(len(split_order)))
    ax.set_xticklabels(split_order, rotation=30, ha="right")
    ax.set_yticks(range(len(row_order)))
    ax.set_yticklabels(row_order, fontsize=7)
    ax.set_title("V11 invariant feature counterfactual name-family diagnostics: mean ratio and within-family std")
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            if pd.notna(values[i, j]):
                ax.text(j, i, f"{values[i, j]:.2f}\nsd {std_values[i, j]:.2f}", ha="center", va="center", fontsize=6, color="black")
    fig.colorbar(im, ax=ax, label="Mean family faithfulness ratio")
    fig.tight_layout()
    if output_path is not None:
        fig.savefig(output_path, dpi=180, bbox_inches="tight")
        print(f"Saved counterfactual family heatmap to {output_path}")
    plt.show()
    return fig, ax

In [ ]:
validation_mask_specs = build_validation_mask_specs(target_sae, feature_rankings, global_soft_gate_specs)
print(f"Built {len(validation_mask_specs)} validation masks.")

validation_results = evaluate_mask_specs_on_splits(
    validation_datasets,
    target_sae,
    validation_mask_specs,
    batch_size=BATCH_SIZE,
)
validation_summary = summarize_validation_results(validation_results)

counterfactual_family_results = pd.DataFrame()
if RUN_COUNTERFACTUAL_DIAGNOSTICS:
    counterfactual_family_results = evaluate_counterfactual_family_diagnostics(
        counterfactual_family_datasets,
        target_sae,
        validation_mask_specs,
        batch_size=BATCH_SIZE,
    )
    validation_summary = add_counterfactual_summary_metrics(validation_summary, counterfactual_family_results)
    counterfactual_family_results.to_csv(COUNTERFACTUAL_DIAGNOSTIC_CSV_PATH, index=False)
    print(f"Saved counterfactual family diagnostics to {COUNTERFACTUAL_DIAGNOSTIC_CSV_PATH}")
else:
    print("RUN_COUNTERFACTUAL_DIAGNOSTICS=False, skipping counterfactual family diagnostics.")

validation_results.to_csv(RESULTS_CSV_PATH, index=False)
validation_summary.to_csv(SUMMARY_CSV_PATH, index=False)
print(f"Saved validation rows to {RESULTS_CSV_PATH}")
print(f"Saved validation summary to {SUMMARY_CSV_PATH}")

summary_cols = [
    "baseline",
    "selection",
    "k",
    "threshold",
    "active_nodes",
    "unique_features",
    "gate_sum",
    "mean_nonzero_gate",
    "opt_mean_faithfulness",
    "opt_max_abs_error",
    "opt_in_band_rate",
    "passes_optimization_splits",
    "cf_train_worst_family_abs_error",
    "cf_train_worst_name_std",
    "cf_train_all_members_in_band_rate",
    "final_mean_faithfulness",
    "final_min_faithfulness",
    "final_max_faithfulness",
    "final_max_abs_error",
    "final_in_band_rate",
    "passes_final_splits",
    "invariant_selection_score",
    "compression_vs_global_500",
]
available_summary_cols = [column for column in summary_cols if column in validation_summary.columns]
print("Top candidates selected by optimization robustness plus invariant counterfactual stability:")
display(validation_summary[available_summary_cols].head(30))

final_sorted = validation_summary.sort_values(
    ["passes_final_splits", "final_max_abs_error", "active_nodes", "final_mean_abs_error"],
    ascending=[False, True, True, True],
).reset_index(drop=True)
print("Best final-validation diagnostic candidates:")
display(final_sorted[available_summary_cols].head(30))

plot_validation_pareto(validation_summary, output_path=PARETO_PLOT_PATH)
plot_generalization_heatmap(validation_results, validation_summary, output_path=HEATMAP_PLOT_PATH, top_n=16)
if RUN_COUNTERFACTUAL_DIAGNOSTICS:
    plot_counterfactual_family_heatmap(counterfactual_family_results, validation_summary, output_path=COUNTERFACTUAL_DIAGNOSTIC_HEATMAP_PATH)

In [ ]:
def select_specs_by_summary(summary, mask_specs, top_n=GROUP_DIAGNOSTIC_TOP_N):
    control_pattern = "random|role_shuffle"
    selected = summary[~summary["baseline"].str.contains(control_pattern, regex=True)].head(top_n)
    selected_ids = set(selected["mask_id"])
    return [spec for spec in mask_specs if spec["mask_id"] in selected_ids]

def short_template_label(template):
    return template.split(",")[0].replace("{subject}", "S").replace("{io}", "IO")[:42]

def evaluate_template_group_diagnostics(split_datasets, sae, mask_specs, splits=None, batch_size=BATCH_SIZE):
    rows = []
    splits = list(splits or FINAL_VALIDATION_SPLITS)
    for split_name in splits:
        dataset = split_datasets[split_name]
        for template_index, (template, group_df) in enumerate(dataset.groupby("template", sort=False)):
            group_df = group_df.reset_index(drop=True)
            if len(group_df) < MIN_GROUP_PROMPTS:
                continue
            full_logits = run_logits(group_df, batch_size=batch_size, detach=True)
            full_logit_diff = mean_logit_diff(full_logits, group_df)
            group_label = f"{split_name}:template_{template_index}"
            print(f"{group_label}: n={len(group_df)}, full logit diff={float(full_logit_diff.item()):.4f}")
            for spec in tqdm(mask_specs, desc=f"Template diagnostics {group_label}"):
                metrics = compute_faithfulness(
                    sae,
                    group_df,
                    feature_mask=spec["feature_mask"],
                    role_feature_mask=spec["role_feature_mask"],
                    full_logit_diff=full_logit_diff,
                    batch_size=batch_size,
                )
                faithfulness = metrics["faithfulness"]
                rows.append({
                    "split": split_name,
                    "group_type": "template",
                    "group_label": group_label,
                    "template_index": template_index,
                    "template": template,
                    "template_short": short_template_label(template),
                    "n_prompts": len(group_df),
                    "mask_id": spec["mask_id"],
                    "baseline": spec["baseline"],
                    "selection": spec["selection"],
                    "k": spec["k"],
                    "threshold": spec["threshold"],
                    "active_nodes": metrics["active_nodes"],
                    "unique_features": metrics["unique_features"],
                    "faithfulness": faithfulness,
                    "faithfulness_abs_error": abs(faithfulness - 1.0),
                    "within_5pct_band": FAITHFULNESS_BAND_LOW <= faithfulness <= FAITHFULNESS_BAND_HIGH,
                    "full_logit_diff": metrics["full_logit_diff"],
                    "masked_logit_diff": metrics["masked_logit_diff"],
                })
    return pd.DataFrame(rows)

def plot_template_group_heatmap(group_results, summary, output_path=None):
    if len(group_results) == 0:
        print("No group diagnostic rows to plot.")
        return None, None
    top_ids = summary.head(GROUP_DIAGNOSTIC_TOP_N)["mask_id"].tolist()
    label_map = {row["mask_id"]: mask_display_label(row) for _, row in summary[summary["mask_id"].isin(top_ids)].iterrows()}
    group_results = group_results[group_results["mask_id"].isin(top_ids)].copy()
    group_results["mask_label"] = group_results["mask_id"].map(label_map)
    group_results["column_label"] = group_results["split"] + "\n" + group_results["template_short"]
    column_order = group_results[["group_label", "column_label"]].drop_duplicates()["column_label"].tolist()
    row_order = [label_map[mask_id] for mask_id in top_ids if mask_id in label_map]
    pivot = group_results.pivot_table(index="mask_label", columns="column_label", values="faithfulness", aggfunc="mean").reindex(index=row_order, columns=column_order)
    values = pivot.to_numpy(dtype=float)

    fig, ax = plt.subplots(figsize=(max(10, 0.75 * len(column_order)), max(4, 0.5 * len(row_order))))
    im = ax.imshow(values, aspect="auto", cmap="coolwarm", vmin=0.65, vmax=1.25)
    ax.set_xticks(range(len(column_order)))
    ax.set_xticklabels(column_order, rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(len(row_order)))
    ax.set_yticklabels(row_order, fontsize=7)
    ax.set_title("V11 invariant feature template-group faithfulness diagnostics for selected masks")
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            if pd.notna(values[i, j]):
                ax.text(j, i, f"{values[i, j]:.2f}", ha="center", va="center", fontsize=6, color="black")
    fig.colorbar(im, ax=ax, label="Faithfulness")
    fig.tight_layout()
    if output_path is not None:
        fig.savefig(output_path, dpi=180, bbox_inches="tight")
        print(f"Saved template group heatmap to {output_path}")
    plt.show()
    return fig, ax

group_diagnostic_results = pd.DataFrame()
if RUN_GROUP_DIAGNOSTICS:
    group_specs = select_specs_by_summary(validation_summary, validation_mask_specs, top_n=GROUP_DIAGNOSTIC_TOP_N)
    print(f"Running template-group diagnostics for {len(group_specs)} selected masks on final validation splits.")
    group_diagnostic_results = evaluate_template_group_diagnostics(
        validation_datasets,
        target_sae,
        group_specs,
        splits=FINAL_VALIDATION_SPLITS,
        batch_size=BATCH_SIZE,
    )
    group_diagnostic_results.to_csv(GROUP_DIAGNOSTIC_CSV_PATH, index=False)
    print(f"Saved template-group diagnostics to {GROUP_DIAGNOSTIC_CSV_PATH}")
    display(group_diagnostic_results.sort_values("faithfulness_abs_error", ascending=False).head(30))
    plot_template_group_heatmap(group_diagnostic_results, validation_summary, output_path=GROUP_DIAGNOSTIC_HEATMAP_PATH)
else:
    print("RUN_GROUP_DIAGNOSTICS=False, skipping template-group diagnostics.")

In [ ]:
manifest_summary_cols = [
    "baseline",
    "selection",
    "k",
    "threshold",
    "active_nodes",
    "unique_features",
    "gate_sum",
    "mean_nonzero_gate",
    "opt_mean_faithfulness",
    "opt_max_abs_error",
    "opt_in_band_rate",
    "passes_optimization_splits",
    "cf_train_worst_family_abs_error",
    "cf_train_worst_name_std",
    "cf_train_all_members_in_band_rate",
    "final_mean_faithfulness",
    "final_min_faithfulness",
    "final_max_faithfulness",
    "final_max_abs_error",
    "final_in_band_rate",
    "passes_final_splits",
    "invariant_selection_score",
    "compression_vs_global_500",
]
manifest_summary_cols = [column for column in manifest_summary_cols if column in validation_summary.columns]

best_rows = validation_summary.head(30)[manifest_summary_cols].to_dict(orient="records")

final_sorted = validation_summary.sort_values(
    ["passes_final_splits", "final_max_abs_error", "active_nodes", "final_mean_abs_error"],
    ascending=[False, True, True, True],
).reset_index(drop=True)
best_final_rows = final_sorted.head(30)[manifest_summary_cols].to_dict(orient="records")

passing_final_rows = validation_summary[validation_summary["passes_final_splits"]].head(30)[manifest_summary_cols].to_dict(orient="records")

top_stable_features = describe_global_features(
    feature_rankings.get("global_stability_vote_calibrated_rescue_union", feature_rankings["global_calibrated_rescue_union"]),
    score_table={
        "activation": train_stats["mean_abs_all"],
        "answer_abs": answer_direction_stats["mean_abs_all"],
        "answer_support": answer_direction_stats["support_all"],
        "stability_vote_rescue": feature_scores.get("global_stability_vote_calibrated_rescue_union", feature_scores["global_calibrated_rescue_union"]),
    },
    limit=50,
).to_dict(orient="records")

counterfactual_summary = {}
if "counterfactual_family_results" in globals() and len(counterfactual_family_results):
    counterfactual_summary = counterfactual_family_results.groupby("split").agg(
        rows=("family_id", "count"),
        families=("family_id", "nunique"),
        mean_family_abs_error=("family_mean_abs_error", "mean"),
        worst_family_abs_error=("family_max_abs_error", "max"),
        mean_name_std=("family_ratio_std", "mean"),
        worst_name_std=("family_ratio_std", "max"),
    ).reset_index().to_dict(orient="records")

manifest = write_run_manifest(
    "completed",
    extra={
        "validation_rows": len(validation_results),
        "summary_rows": len(validation_summary),
        "stability_trace_rows": len(stability_trace),
        "group_diagnostic_rows": len(group_diagnostic_results),
        "counterfactual_family_rows": len(counterfactual_family_results) if "counterfactual_family_results" in globals() else 0,
        "counterfactual_family_summary": counterfactual_summary,
        "top_stable_features": top_stable_features,
        "top_invariant_selected_candidates": best_rows,
        "top_final_validation_candidates": best_final_rows,
        "passing_final_split_candidates": passing_final_rows,
    },
)

copied = mirror_artifacts_to_latest([
    SMOKE_RESULTS_CSV_PATH,
    SMOKE_PLOT_PATH,
    RESULTS_CSV_PATH,
    SUMMARY_CSV_PATH,
    PARETO_PLOT_PATH,
    HEATMAP_PLOT_PATH,
    GROUP_DIAGNOSTIC_CSV_PATH,
    GROUP_DIAGNOSTIC_HEATMAP_PATH,
    COUNTERFACTUAL_DIAGNOSTIC_CSV_PATH,
    COUNTERFACTUAL_DIAGNOSTIC_HEATMAP_PATH,
    INVARIANT_TRACE_CSV_PATH,
    MASKS_PATH,
    MANIFEST_PATH,
])
print("Completed run manifest:")
print(json.dumps(manifest, indent=2))
print("Mirrored latest artifacts:")
for path in copied:
    print(path)

## 12. Expected Artifacts and Reading the Result

Expected Drive layout:

```text
MyDrive/minimum_sae_circuit_discovery/
  cache/v011_invariant_feature_circuit_benchmark/
    pooled_train_role_feature_stats_layer8_ioi.pt
    pooled_train_answer_direction_role_feature_stats_layer8_ioi.pt
    train_core_role_feature_stats_layer8_ioi.pt
    train_core_answer_direction_role_feature_stats_layer8_ioi.pt
    ...
  runs/v011_invariant_feature_circuit_benchmark/trial_YYYYMMDD_HHMMSS/
    run_manifest.json
    smoke_invariant_feature_results.csv
    smoke_invariant_feature_pareto.png
    invariant_feature_validation_results.csv
    invariant_feature_validation_summary.csv
    invariant_feature_validation_pareto.png
    invariant_feature_generalization_heatmap.png
    invariant_feature_template_group_diagnostics.csv
    invariant_feature_template_group_heatmap.png
    invariant_feature_family_diagnostics.csv
    invariant_feature_family_heatmap.png
    invariant_feature_stability_trace.csv
    selected_masks.pt
```

Read the result as a benchmark, not just a leaderboard:

1. If `global_env_*` masks beat pooled masks on their source split but fail final held-out names/templates, the old minimum-circuit target is overfitting.
2. If `global_stability_*` masks improve final worst-case error or name-family variance, stability selection is useful.
3. If stable masks need much larger K to pass, the publishable claim is that minimum faithful circuits become larger once invariance is required.
4. If no stable single-layer SAE mask passes final splits, the next publishable step is cross-layer transcoders or attribution graphs rather than another layer-8 ranking tweak.